In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

files = {
    "metric.csv": "kpi_name",
    "trace.csv": "trace_name",
    "log.csv": "log_name",
    "error_logs.csv": "message"
}

rows = []
sample_rows_dict = {}

for fname, name_col in files.items():
    try:
        df = pd.read_csv(fname)
    except FileNotFoundError:
        # create a compact record for missing files
        rows.append({
            "filename": fname,
            "n_rows": 0,
            "n_unique_cmdb_id": 0,
            "min_ts": pd.NaT,
            "max_ts": pd.NaT,
            "sample_names": [],
        })
        sample_rows_dict[fname] = pd.DataFrame()
        continue

    n_rows = len(df)
    n_unique_cmdb_id = int(df['cmdb_id'].nunique()) if 'cmdb_id' in df.columns else 0

    # timestamps: convert epoch seconds -> aware datetimes in Asia/Shanghai
    if 'timestamp' in df.columns:
        ts_series = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        min_ts = ts_series.min()
        max_ts = ts_series.max()
        # format compactly
        min_ts_str = min_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
        max_ts_str = max_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        min_ts_str = pd.NaT
        max_ts_str = pd.NaT

    # sample names (unique KPI/trace/log names or message). list up to 50.
    if name_col in df.columns:
        unique_names = list(pd.Index(df[name_col].dropna().unique())[:50])
    else:
        unique_names = []

    # sample rows: select a small set of relevant columns and head(10)
    cols = []
    if 'timestamp' in df.columns:
        cols.append('timestamp')
    if 'cmdb_id' in df.columns:
        cols.append('cmdb_id')
    if name_col in df.columns:
        cols.append(name_col)
    # include value column if present
    if 'value' in df.columns:
        cols.append('value')

    # ensure unique and valid
    cols = [c for c in cols if c in df.columns]
    sample_df = df[cols].head(10).copy()
    # convert timestamp column to localized string for compact display if exists
    if 'timestamp' in sample_df.columns:
        sample_df['timestamp'] = pd.to_datetime(sample_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    rows.append({
        "filename": fname,
        "n_rows": n_rows,
        "n_unique_cmdb_id": n_unique_cmdb_id,
        "min_ts": min_ts_str,
        "max_ts": max_ts_str,
        "sample_names": unique_names
    })
    sample_rows_dict[fname] = sample_df

summary_df = pd.DataFrame(rows)[["filename","n_rows","n_unique_cmdb_id","min_ts","max_ts","sample_names"]]

# display compact summary table and the small sample rows per file
summary_df, sample_rows_dict

```
Out[1]:
```
```python
# Summarize the previously computed file overviews in plain English.
# Reuse the existing 'summary_df' variable from prior execution if available.
if 'summary_df' in globals():
    df = summary_df
else:
    import pandas as pd
    df = pd.DataFrame([
        {"filename":"metric.csv","n_rows":145812,"n_unique_cmdb_id":59,"min_ts":"2022-03-21 00:00:00 CST","max_ts":"2022-03-21 00:29:00 CST","sample_names":[]},
        {"filename":"trace.csv","n_rows":29376,"n_unique_cmdb_id":40,"min_ts":"2022-03-21 00:00:00 CST","max_ts":"2022-03-21 00:29:00 CST","sample_names":[]},
        {"filename":"log.csv","n_rows":1748,"n_unique_cmdb_id":31,"min_ts":"2022-03-21 00:00:00 CST","max_ts":"2022-03-21 00:29:00 CST","sample_names":[]},
        {"filename":"error_logs.csv","n_rows":540,"n_unique_cmdb_id":3,"min_ts":"2022-03-21 00:00:05 CST","max_ts":"2022-03-21 00:29:59 CST","sample_names":[]}
    ])

parts = []
for _, r in df.iterrows():
    names = r.get("sample_names", []) or []
    # show up to 10 sample names for compactness
    names_shown = ", ".join([str(x) for x in names[:10]])
    if len(names) > 10:
        names_shown += ", ..."
    parts.append(
        f"{r['filename']}: {int(r['n_rows'])} rows, {int(r['n_unique_cmdb_id'])} unique cmdb_id(s), "
        f"time range {r['min_ts']} → {r['max_ts']}. Sample names (up to 10): [{names_shown}]"
    )

summary = "Summary of telemetry files:\n" + "\n".join(parts) + (
    "\n\nNote: For each file the first up to 10 sample rows were displayed earlier; full time series are not shown here."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         filename  n_rows  n_unique_cmdb_id                   min_ts                   max_ts                                       sample_names
0      metric.csv  145812                59  2022-03-21 00:00:00 CST  2022-03-21 00:29:00 CST  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...
1       trace.csv   29376                40  2022-03-21 00:00:00 CST  2022-03-21 00:29:00 CST  [trace.from_frontend-0.duration_mean, trace.fr...
2         log.csv    1748                31  2022-03-21 00:00:00 CST  2022-03-21 00:29:00 CST                   [log.error_count, log.row_count]
3  error_logs.csv     540                 3  2022-03-21 00:00:05 CST  2022-03-21 00:29:59 CST  [severity: warning, message: failed to retriev..., {'metric.csv':                  timestamp    cmdb_id                                           kpi_name        value
0  2022-03-21 00:00:00 CST  adservice                                     app.grpc.count   111.000000
1  2022-03-21 00:00:00 CST  adservice                                       app.grpc.mrt     0.002779
2  2022-03-21 00:00:00 CST  adservice                                        app.grpc.rr   100.000000
3  2022-03-21 00:00:00 CST  adservice                                        app.grpc.sr    76.576577
4  2022-03-21 00:00:00 CST  adservice                                     app.http.count    24.000000
5  2022-03-21 00:00:00 CST  adservice                                       app.http.mrt     0.010150
6  2022-03-21 00:00:00 CST  adservice                                        app.http.rr   100.000000
7  2022-03-21 00:00:00 CST  adservice                                        app.http.sr   100.000000
8  2022-03-21 00:00:00 CST  adservice    runtime.java_lang_ClassLoading_LoadedClassCount  4990.000000
9  2022-03-21 00:00:00 CST  adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...  4990.000000, 'trace.csv':                  timestamp      cmdb_id                           trace_name      value
0  2022-03-21 00:00:00 CST  adservice-0  trace.from_frontend-0.duration_mean   0.000015
1  2022-03-21 00:00:00 CST  adservice-0   trace.from_frontend-0.duration_p95   0.000017
2  2022-03-21 00:00:00 CST  adservice-0     trace.from_frontend-0.error_rate   0.000000
3  2022-03-21 00:00:00 CST  adservice-0      trace.from_frontend-0.row_count  10.000000
4  2022-03-21 00:00:00 CST  adservice-0  trace.from_frontend-1.duration_mean   0.000013
5  2022-03-21 00:00:00 CST  adservice-0   trace.from_frontend-1.duration_p95   0.000015
6  2022-03-21 00:00:00 CST  adservice-0     trace.from_frontend-1.error_rate   0.000000
7  2022-03-21 00:00:00 CST  adservice-0      trace.from_frontend-1.row_count   7.000000
8  2022-03-21 00:00:00 CST  adservice-0  trace.from_frontend-2.duration_mean   0.000013
9  2022-03-21 00:00:00 CST  adservice-0   trace.from_frontend-2.duration_p95   0.000016, 'log.csv':                  timestamp        cmdb_id         log_name  value
0  2022-03-21 00:00:00 CST    adservice-0  log.error_count    0.0
1  2022-03-21 00:00:00 CST    adservice-0    log.row_count  518.0
2  2022-03-21 00:00:00 CST    adservice-1  log.error_count    0.0
3  2022-03-21 00:00:00 CST    adservice-1    log.row_count   46.0
4  2022-03-21 00:00:00 CST    adservice-2  log.error_count    0.0
5  2022-03-21 00:00:00 CST    adservice-2    log.row_count   46.0
6  2022-03-21 00:00:00 CST  cartservice-0  log.error_count    0.0
7  2022-03-21 00:00:00 CST  cartservice-0    log.row_count  272.0
8  2022-03-21 00:00:00 CST  cartservice-1  log.error_count    0.0
9  2022-03-21 00:00:00 CST  cartservice-1    log.row_count  253.0, 'error_logs.csv':                  timestamp     cmdb_id                                            message
0  2022-03-21 00:00:05 CST  frontend-2  severity: warning, message: failed to retrieve...
1  2022-03-21 00:00:11 CST  frontend-0  severity: warning, message: failed to retrieve...
2  2022-03-21 00:00:12 CST  frontend-1  severity: warning, message: failed to retrieve...
3  2022-03-21 00:00:13 CST  frontend-2  severity: warning, message: failed to retrieve...
4  2022-03-21 00:00:13 CST  frontend-0  severity: warning, message: failed to retrieve...
5  2022-03-21 00:00:19 CST  frontend-1  severity: warning, message: failed to retrieve...
6  2022-03-21 00:00:21 CST  frontend-0  severity: warning, message: failed to retrieve...
7  2022-03-21 00:00:25 CST  frontend-1  severity: warning, message: failed to retrieve...
8  2022-03-21 00:00:26 CST  frontend-2  severity: warning, message: failed to retrieve...
9  2022-03-21 00:00:26 CST  frontend-0  severity: warning, message: failed to retrieve...})```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load datasets (reuse variable names if already present)
metric_df = pd.read_csv("metric.csv") if 'metric_df' not in globals() else metric_df
trace_df = pd.read_csv("trace.csv") if 'trace_df' not in globals() else trace_df
log_df = pd.read_csv("log.csv") if 'log_df' not in globals() else log_df
error_df = pd.read_csv("error_logs.csv") if 'error_df' not in globals() else error_df

# Ensure numeric 'value' where applicable
for df in (metric_df, trace_df, log_df):
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) metric.csv: group by cmdb_id and kpi_name -> count, p95, p5, mean, max
metric_grp = metric_df.dropna(subset=['value']).groupby(['cmdb_id', 'kpi_name'])['value']
metric_agg = metric_grp.agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05),
    mean='mean',
    max='max'
).reset_index()
metric_top20 = metric_agg.sort_values('count', ascending=False).head(20)
metric_totals = {
    'distinct_kpi_name': int(metric_df['kpi_name'].nunique()),
    'distinct_cmdb_id': int(metric_df['cmdb_id'].nunique())
}

# 2) trace.csv: group by cmdb_id and trace_name -> count, p95, p5, mean, max
trace_grp = trace_df.dropna(subset=['value']).groupby(['cmdb_id', 'trace_name'])['value']
trace_agg = trace_grp.agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05),
    mean='mean',
    max='max'
).reset_index()
trace_top20 = trace_agg.sort_values('count', ascending=False).head(20)
trace_totals = {
    'distinct_trace_name': int(trace_df['trace_name'].nunique()),
    'distinct_cmdb_id': int(trace_df['cmdb_id'].nunique())
}

# 3) log.csv: group by cmdb_id and log_name -> count, p95, p5, mean, max (value)
log_grp = log_df.dropna(subset=['value']).groupby(['cmdb_id', 'log_name'])['value']
log_agg = log_grp.agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05),
    mean='mean',
    max='max'
).reset_index()
log_top20 = log_agg.sort_values('count', ascending=False).head(20)
log_totals = {
    'distinct_log_name': int(log_df['log_name'].nunique()),
    'distinct_cmdb_id': int(log_df['cmdb_id'].nunique())
}

# 4) error_logs.csv: counts per cmdb_id, earliest timestamp per cmdb_id, and up to 20 rows ordered by timestamp asc
# Convert timestamp to localized datetime for error logs
error_df = error_df.copy()
if 'timestamp' in error_df.columns:
    error_df['_ts_dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    error_df['_ts_dt'] = pd.NaT

error_counts = error_df.groupby('cmdb_id').agg(
    error_count=('message', 'count'),
    earliest_ts=('_ts_dt', 'min')
).reset_index()
# Format earliest timestamp compactly
error_counts['earliest_ts_str'] = error_counts['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Sample up to 20 error rows ordered by timestamp asc (compact)
error_sample_rows = error_df.sort_values('_ts_dt', ascending=True).head(20).copy()
if '_ts_dt' in error_sample_rows.columns:
    error_sample_rows['timestamp'] = error_sample_rows['_ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
# Select only required columns and limit to 20 rows
error_sample_rows = error_sample_rows[['timestamp', 'cmdb_id', 'message']].head(20)

# Keep outputs compact as required
metric_top20, metric_totals, trace_top20, trace_totals, log_top20, log_totals, error_counts[['cmdb_id','error_count','earliest_ts_str']].head(20), error_sample_rows

```
Out[2]:
```
```python
# Compose a plain-English compact summary using previously computed aggregates.
# Reuse existing variables from the kernel: metric_top20, metric_totals, trace_top20, trace_totals,
# log_top20, log_totals, error_counts, error_sample_rows.

# Fallback: if some variables are missing, load minimal info from files.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

if 'metric_top20' not in globals():
    metric_df = pd.read_csv("metric.csv")
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    metric_grp = metric_df.dropna(subset=['value']).groupby(['cmdb_id','kpi_name'])['value']
    metric_top20 = metric_grp.agg(count='count', p95=lambda x: x.quantile(0.95),
                                  p5=lambda x: x.quantile(0.05), mean='mean', max='max').reset_index().sort_values('count',ascending=False).head(20)
    metric_totals = {'distinct_kpi_name': int(metric_df['kpi_name'].nunique()), 'distinct_cmdb_id': int(metric_df['cmdb_id'].nunique())}

if 'trace_top20' not in globals():
    trace_df = pd.read_csv("trace.csv")
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
    trace_grp = trace_df.dropna(subset=['value']).groupby(['cmdb_id','trace_name'])['value']
    trace_top20 = trace_grp.agg(count='count', p95=lambda x: x.quantile(0.95),
                                p5=lambda x: x.quantile(0.05), mean='mean', max='max').reset_index().sort_values('count',ascending=False).head(20)
    trace_totals = {'distinct_trace_name': int(trace_df['trace_name'].nunique()), 'distinct_cmdb_id': int(trace_df['cmdb_id'].nunique())}

if 'log_top20' not in globals():
    log_df = pd.read_csv("log.csv")
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
    log_grp = log_df.dropna(subset=['value']).groupby(['cmdb_id','log_name'])['value']
    log_top20 = log_grp.agg(count='count', p95=lambda x: x.quantile(0.95),
                             p5=lambda x: x.quantile(0.05), mean='mean', max='max').reset_index().sort_values('count',ascending=False).head(20)
    log_totals = {'distinct_log_name': int(log_df['log_name'].nunique()), 'distinct_cmdb_id': int(log_df['cmdb_id'].nunique())}

if 'error_counts' not in globals():
    error_df = pd.read_csv("error_logs.csv")
    error_df['_ts_dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    error_counts = error_df.groupby('cmdb_id').agg(error_count=('message','count'), earliest_ts=('_ts_dt','min')).reset_index()
    error_counts['earliest_ts_str'] = error_counts['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_sample_rows = error_df.sort_values('_ts_dt').head(20)
    error_sample_rows['timestamp'] = error_sample_rows['_ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_sample_rows = error_sample_rows[['timestamp','cmdb_id','message']].head(20)

# Prepare concise examples for each dataset
def fmt_row(r):
    return f"{r['cmdb_id']} / {r.iloc[1]} (count={int(r['count'])}, p95={r['p95']:.6g}, p5={r['p5']:.6g}, mean={r['mean']:.6g}, max={r['max']:.6g})"

# Example rows (first two) for metric, trace, log if available
metric_examples = []
if len(metric_top20) >= 1:
    r = metric_top20.iloc[0]
    metric_examples.append(f"{r['cmdb_id']} / {r['kpi_name']} (count={int(r['count'])}, p95={r['p95']:.6g}, p5={r['p5']:.6g}, mean={r['mean']:.6g}, max={r['max']:.6g})")
if len(metric_top20) >= 2:
    r = metric_top20.iloc[1]
    metric_examples.append(f"{r['cmdb_id']} / {r['kpi_name']} (count={int(r['count'])}, p95={r['p95']:.6g}, p5={r['p5']:.6g}, mean={r['mean']:.6g}, max={r['max']:.6g})")

trace_examples = []
if len(trace_top20) >= 1:
    r = trace_top20.iloc[0]
    trace_examples.append(f"{r['cmdb_id']} / {r['trace_name']} (count={int(r['count'])}, p95={r['p95']:.6g}, p5={r['p5']:.6g}, mean={r['mean']:.6g}, max={r['max']:.6g})")
if len(trace_top20) >= 2:
    r = trace_top20.iloc[1]
    trace_examples.append(f"{r['cmdb_id']} / {r['trace_name']} (count={int(r['count'])}, p95={r['p95']:.6g}, p5={r['p5']:.6g}, mean={r['mean']:.6g}, max={r['max']:.6g})")

log_examples = []
if len(log_top20) >= 1:
    r = log_top20.iloc[0]
    log_examples.append(f"{r['cmdb_id']} / {r['log_name']} (count={int(r['count'])}, p95={r['p95']:.6g}, p5={r['p5']:.6g}, mean={r['mean']:.6g}, max={r['max']:.6g})")
if len(log_top20) >= 2:
    r = log_top20.iloc[1]
    log_examples.append(f"{r['cmdb_id']} / {r['log_name']} (count={int(r['count'])}, p95={r['p95']:.6g}, p5={r['p5']:.6g}, mean={r['mean']:.6g}, max={r['max']:.6g})")

# Top error log counts (top 3)
top_errors = error_counts.sort_values('error_count', ascending=False).head(3)
top_errors_list = [f"{row['cmdb_id']}: {int(row['error_count'])} rows, earliest {row['earliest_ts_str']}" for _, row in top_errors.iterrows()]

summary = (
    "Compact summary of telemetry aggregates:\n\n"
    f"1) metric.csv: total rows = {int(metric_top20['count'].sum()) if False else 'see counts per series'}, "
    f"distinct_kpi_name = {metric_totals.get('distinct_kpi_name')}, distinct_cmdb_id = {metric_totals.get('distinct_cmdb_id')}.\n"
    "   Top 20 component-KPI pairs (sorted by count) mostly have count = 30. Examples:\n"
    f"   - {metric_examples[0] if metric_examples else 'N/A'}\n"
    f"   - {metric_examples[1] if len(metric_examples)>1 else 'N/A'}\n\n"
    f"2) trace.csv: distinct_trace_name = {trace_totals.get('distinct_trace_name')}, distinct_cmdb_id = {trace_totals.get('distinct_cmdb_id')}.\n"
    "   Top trace series (by count) also have count = 30. Examples:\n"
    f"   - {trace_examples[0] if trace_examples else 'N/A'}\n"
    f"   - {trace_examples[1] if len(trace_examples)>1 else 'N/A'}\n\n"
    f"3) log.csv: distinct_log_name = {log_totals.get('distinct_log_name')}, distinct_cmdb_id = {log_totals.get('distinct_cmdb_id')}.\n"
    "   Two main log metrics are present (log.error_count and log.row_count). Examples:\n"
    f"   - {log_examples[0] if log_examples else 'N/A'}\n"
    f"   - {log_examples[1] if len(log_examples)>1 else 'N/A'}\n\n"
    "4) error_logs.csv: error counts per cmdb_id (top 3):\n"
    f"   - {top_errors_list[0] if len(top_errors_list)>0 else 'N/A'}\n"
    f"   - {top_errors_list[1] if len(top_errors_list)>1 else 'N/A'}\n"
    f"   - {top_errors_list[2] if len(top_errors_list)>2 else 'N/A'}\n"
    "   Also, up to 20 earliest error rows (timestamp, cmdb_id, message) were prepared and shown earlier.\n\n"
    "Notes:\n"
    "- Per-series statistics (count, p95, p5, mean, max) were computed globally using the full series for each component-KPI/trace/log.\n"
    "- The metric dataset contains many KPIs (≈1176) across 59 components; trace has ≈200 trace names across 40 components; log has 2 log names across 31 components.\n"
    "- Full top-20 tables and the error log sample were produced earlier and can be inspected for exact numeric detail."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  count           p95            p5          mean           max
4869  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30  1.171612e+03  3.208875e+02  7.794167e+02  1.243750e+03
0              adservice                                     app.grpc.count     30  1.092000e+02  5.990000e+01  7.796667e+01  1.110000e+02
1              adservice                                       app.grpc.mrt     30  2.890461e-03  2.500473e-03  2.704459e-03  2.975701e-03
2              adservice                                        app.grpc.rr     30  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
3              adservice                                        app.grpc.sr     30  8.108108e+01  7.361449e+01  7.657005e+01  8.115942e+01
4              adservice                                     app.http.count     30  2.400000e+01  2.400000e+01  2.400000e+01  2.400000e+01
4830  shippingservice2-0  container.node-5.container_network_receive_MB....     30  1.747506e-01  2.275174e-02  4.535421e-02  2.932096e-01
4831  shippingservice2-0  container.node-5.container_network_receive_err...     30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4832  shippingservice2-0  container.node-5.container_network_receive_pac...     30  3.398500e+02  2.554000e+02  2.936333e+02  3.490000e+02
4833  shippingservice2-0  container.node-5.container_network_receive_pac...     30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4834  shippingservice2-0  container.node-5.container_network_transmit_MB...     30  5.582952e-01  3.436612e-01  4.508038e-01  7.121944e-01
4835  shippingservice2-0  container.node-5.container_network_transmit_er...     30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4836  shippingservice2-0  container.node-5.container_network_transmit_pa...     30  2.491250e+02  1.722250e+02  2.029667e+02  2.745000e+02
4837  shippingservice2-0  container.node-5.container_network_transmit_pa...     30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4822  shippingservice2-0  container.node-5.container_memory_failures.hie...     30  5.507500e+01  0.000000e+00  2.243333e+01  9.100000e+01
4823  shippingservice2-0  container.node-5.container_memory_failures.hie...     30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4824  shippingservice2-0      container.node-5.container_memory_mapped_file     30  4.915200e+05  4.915200e+05  4.915200e+05  4.915200e+05
4825  shippingservice2-0     container.node-5.container_memory_max_usage_MB     30  1.280117e+02  1.280117e+02  1.280117e+02  1.280117e+02
4826  shippingservice2-0              container.node-5.container_memory_rss     30  1.471590e+07  1.406566e+07  1.438222e+07  1.476198e+07
4827  shippingservice2-0             container.node-5.container_memory_swap     30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00, {'distinct_kpi_name': 1176, 'distinct_cmdb_id': 59},                        cmdb_id                                      trace_name  count        p95         p5       mean        max
1144  recommendationservice2-0  trace.to_productcatalogservice-1.duration_mean     30   0.000028   0.000023   0.000025   0.000029
1145  recommendationservice2-0   trace.to_productcatalogservice-1.duration_p95     30   0.000044   0.000026   0.000031   0.000045
1146  recommendationservice2-0     trace.to_productcatalogservice-1.error_rate     30   0.000000   0.000000   0.000000   0.000000
1147  recommendationservice2-0      trace.to_productcatalogservice-1.row_count     30  10.000000   7.450000   8.966667  10.000000
1148  recommendationservice2-0  trace.to_productcatalogservice-2.duration_mean     30   0.000029   0.000024   0.000026   0.000029
1149  recommendationservice2-0   trace.to_productcatalogservice-2.duration_p95     30   0.000042   0.000027   0.000033   0.000047
1150  recommendationservice2-0     trace.to_productcatalogservice-2.error_rate     30   0.000000   0.000000   0.000000   0.000000
1151  recommendationservice2-0      trace.to_productcatalogservice-2.row_count     30  10.550000   7.000000   9.000000  11.000000
1136  recommendationservice2-0                        trace.self.duration_mean     30   0.002414   0.002244   0.002318   0.002425
1137  recommendationservice2-0                         trace.self.duration_p95     30   0.002959   0.002576   0.002754   0.003067
1138  recommendationservice2-0                           trace.self.error_rate     30   0.000000   0.000000   0.000000   0.000000
1139  recommendationservice2-0                            trace.self.row_count     30  31.000000  21.450000  26.966667  32.000000
1140  recommendationservice2-0  trace.to_productcatalogservice-0.duration_mean     30   0.000029   0.000023   0.000025   0.000029
1141  recommendationservice2-0   trace.to_productcatalogservice-0.duration_p95     30   0.000042   0.000025   0.000031   0.000046
1142  recommendationservice2-0     trace.to_productcatalogservice-0.error_rate     30   0.000000   0.000000   0.000000   0.000000
1143  recommendationservice2-0      trace.to_productcatalogservice-0.row_count     30  10.550000   7.000000   9.000000  11.000000
1128   recommendationservice-2  trace.to_productcatalogservice-2.duration_mean     30   0.000027   0.000022   0.000024   0.000027
1129   recommendationservice-2   trace.to_productcatalogservice-2.duration_p95     30   0.000037   0.000024   0.000029   0.000041
1130   recommendationservice-2     trace.to_productcatalogservice-2.error_rate     30   0.000000   0.000000   0.000000   0.000000
1131   recommendationservice-2      trace.to_productcatalogservice-2.row_count     30   8.000000   6.000000   7.100000   9.000000, {'distinct_trace_name': 200, 'distinct_cmdb_id': 40},                     cmdb_id         log_name  count     p95      p5        mean    max
0               adservice-0  log.error_count     30    0.00    0.00    0.000000    0.0
1               adservice-0    log.row_count     30  517.10  330.80  408.200000  538.0
2               adservice-1  log.error_count     30    0.00    0.00    0.000000    0.0
3               adservice-1    log.row_count     30   46.00   30.00   36.133333   46.0
4               adservice-2  log.error_count     30    0.00    0.00    0.000000    0.0
5               adservice-2    log.row_count     30   44.20   30.00   36.066667   46.0
6             cartservice-0  log.error_count     30    0.00    0.00    0.000000    0.0
7             cartservice-0    log.row_count     30  328.00  246.05  285.466667  329.0
8             cartservice-1  log.error_count     30    0.00    0.00    0.000000    0.0
9             cartservice-1    log.row_count     30  329.00  249.35  285.766667  351.0
10            cartservice-2  log.error_count     30    0.00    0.00    0.000000    0.0
11            cartservice-2    log.row_count     30  325.50  240.80  285.133333  332.0
20        currencyservice-1  log.error_count     30    0.00    0.00    0.000000    0.0
21        currencyservice-1    log.row_count     30  282.30  216.60  249.000000  320.0
23        currencyservice-2    log.row_count     30  282.75  217.90  249.466667  322.0
22        currencyservice-2  log.error_count     30    0.00    0.00    0.000000    0.0
19        currencyservice-0    log.row_count     30  282.65  215.25  250.166667  321.0
18        currencyservice-0  log.error_count     30    0.00    0.00    0.000000    0.0
50  recommendationservice-1  log.error_count     30    0.00    0.00    0.000000    0.0
51  recommendationservice-1    log.row_count     30   84.55   65.35   75.166667   90.0, {'distinct_log_name': 2, 'distinct_cmdb_id': 31},       cmdb_id  error_count          earliest_ts_str
0  frontend-0          229  2022-03-21 00:00:11 CST
1  frontend-1          153  2022-03-21 00:00:12 CST
2  frontend-2          158  2022-03-21 00:00:05 CST,                   timestamp     cmdb_id                                            message
0   2022-03-21 00:00:05 CST  frontend-2  severity: warning, message: failed to retrieve...
1   2022-03-21 00:00:11 CST  frontend-0  severity: warning, message: failed to retrieve...
2   2022-03-21 00:00:12 CST  frontend-1  severity: warning, message: failed to retrieve...
3   2022-03-21 00:00:13 CST  frontend-2  severity: warning, message: failed to retrieve...
4   2022-03-21 00:00:13 CST  frontend-0  severity: warning, message: failed to retrieve...
5   2022-03-21 00:00:19 CST  frontend-1  severity: warning, message: failed to retrieve...
6   2022-03-21 00:00:21 CST  frontend-0  severity: warning, message: failed to retrieve...
7   2022-03-21 00:00:25 CST  frontend-1  severity: warning, message: failed to retrieve...
8   2022-03-21 00:00:26 CST  frontend-2  severity: warning, message: failed to retrieve...
9   2022-03-21 00:00:26 CST  frontend-0  severity: warning, message: failed to retrieve...
10  2022-03-21 00:00:30 CST  frontend-0  severity: warning, message: failed to retrieve...
11  2022-03-21 00:00:32 CST  frontend-1  severity: warning, message: failed to retrieve...
12  2022-03-21 00:00:36 CST  frontend-0  severity: warning, message: failed to retrieve...
13  2022-03-21 00:00:39 CST  frontend-1  severity: warning, message: failed to retrieve...
14  2022-03-21 00:00:39 CST  frontend-2  severity: warning, message: failed to retrieve...
15  2022-03-21 00:00:41 CST  frontend-0  severity: warning, message: failed to retrieve...
16  2022-03-21 00:00:45 CST  frontend-1  severity: warning, message: failed to retrieve...
17  2022-03-21 00:00:47 CST  frontend-0  severity: warning, message: failed to retrieve...
18  2022-03-21 00:00:50 CST  frontend-2  severity: warning, message: failed to retrieve...
19  2022-03-21 00:00:55 CST  frontend-0  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing loaded DataFrames if available, otherwise load from CSVs
metric_df = metric_df if 'metric_df' in globals() else pd.read_csv("metric.csv")
trace_df = trace_df if 'trace_df' in globals() else pd.read_csv("trace.csv")
log_df = log_df if 'log_df' in globals() else pd.read_csv("log.csv")
error_df = error_df if 'error_df' in globals() else pd.read_csv("error_logs.csv")

# Ensure numeric 'value' columns
for df in (metric_df, trace_df, log_df):
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) Compute global thresholds (P95, P5) per series BEFORE filtering (metric, trace, log)
if 'metric_thresh' in globals():
    metric_thresh = metric_thresh
else:
    metric_thresh = (
        metric_df.dropna(subset=['value'])
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
    )

if 'trace_thresh' in globals():
    trace_thresh = trace_thresh
else:
    trace_thresh = (
        trace_df.dropna(subset=['value'])
        .groupby(['cmdb_id', 'trace_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
    )

if 'log_thresh' in globals():
    log_thresh = log_thresh
else:
    log_thresh = (
        log_df.dropna(subset=['value'])
        .groupby(['cmdb_id', 'log_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
    )

# 2) Define incident window in epoch seconds using Asia/Shanghai tz
start_dt = pd.Timestamp("2022-03-21 00:00:00").tz_localize(tz)
end_dt = pd.Timestamp("2022-03-21 00:30:00").tz_localize(tz)
# Convert to UTC epoch seconds since stored timestamps are epoch seconds (UTC-based)
# But safe approach: convert epoch seconds column to localized datetime and filter by that.
def add_dt_column(df):
    if '_ts_dt' not in df.columns:
        df['_ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

metric_df = add_dt_column(metric_df)
trace_df = add_dt_column(trace_df)
log_df = add_dt_column(log_df)
error_df = add_dt_column(error_df)

# Filter to window (inclusive start, exclusive end)
metric_win = metric_df[(metric_df['_ts_dt'] >= start_dt) & (metric_df['_ts_dt'] < end_dt)].copy()
trace_win = trace_df[(trace_df['_ts_dt'] >= start_dt) & (trace_df['_ts_dt'] < end_dt)].copy()
log_win = log_df[(log_df['_ts_dt'] >= start_dt) & (log_df['_ts_dt'] < end_dt)].copy()
error_win = error_df[(error_df['_ts_dt'] >= start_dt) & (error_df['_ts_dt'] < end_dt)].copy()

# 3) Detect anomalies: value >= global_p95 OR value <= global_p5
# Merge thresholds then flag anomalies
metric_win = metric_win.merge(metric_thresh, on=['cmdb_id', 'kpi_name'], how='left')
metric_win['is_anom'] = metric_win.apply(
    lambda r: False if pd.isna(r['value']) or pd.isna(r['p95']) or pd.isna(r['p5']) else (r['value'] >= r['p95'] or r['value'] <= r['p5']),
    axis=1
)

trace_win = trace_win.merge(trace_thresh, on=['cmdb_id', 'trace_name'], how='left')
trace_win['is_anom'] = trace_win.apply(
    lambda r: False if pd.isna(r['value']) or pd.isna(r['p95']) or pd.isna(r['p5']) else (r['value'] >= r['p95'] or r['value'] <= r['p5']),
    axis=1
)

log_win = log_win.merge(log_thresh, on=['cmdb_id', 'log_name'], how='left')
log_win['is_anom'] = log_win.apply(
    lambda r: False if pd.isna(r['value']) or pd.isna(r['p95']) or pd.isna(r['p5']) else (r['value'] >= r['p95'] or r['value'] <= r['p5']),
    axis=1
)

# 4) Aggregate anomalies per series: anomaly_count, earliest_anomaly_timestamp, in_window_max_value, in_window_min_value
def aggregate_anomalies(df, group_cols, ts_col='_ts_dt'):
    # full intermediate stored as *_anom_full if needed
    anom_df = df[df['is_anom']].copy()
    if anom_df.empty:
        return pd.DataFrame(columns=group_cols + ['anomaly_count', 'earliest_anomaly_timestamp', 'in_window_max_value', 'in_window_min_value'])
    agg = (
        anom_df.groupby(group_cols)
        .agg(
            anomaly_count=('is_anom', 'sum'),
            earliest_anomaly_timestamp=(ts_col, 'min'),
            in_window_max_value=('value', 'max'),
            in_window_min_value=('value', 'min')
        )
        .reset_index()
    )
    # format earliest timestamp to string in Asia/Shanghai
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    return agg

metric_anom_agg = aggregate_anomalies(metric_win, ['cmdb_id', 'kpi_name'])
trace_anom_agg = aggregate_anomalies(trace_win, ['cmdb_id', 'trace_name'])
log_anom_agg = aggregate_anomalies(log_win, ['cmdb_id', 'log_name'])

# Sort and keep top 20 by anomaly_count desc, filter anomaly_count>0 already done in aggregation
metric_anom_top20 = metric_anom_agg.sort_values('anomaly_count', ascending=False).head(20)
trace_anom_top20 = trace_anom_agg.sort_values('anomaly_count', ascending=False).head(20)
log_anom_top20 = log_anom_agg.sort_values('anomaly_count', ascending=False).head(20)

# 5) error_logs in-window counts per cmdb_id and earliest timestamp, plus up to 20 earliest raw rows
if not error_win.empty:
    error_counts_window = (
        error_win.groupby('cmdb_id')
        .agg(error_count=('message', 'count'), earliest_ts=('_ts_dt', 'min'))
        .reset_index()
    )
    error_counts_window['earliest_ts'] = error_counts_window['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_counts_window = error_counts_window.sort_values('error_count', ascending=False).head(20)
    error_sample_window = (
        error_win.sort_values('_ts_dt', ascending=True)
        .head(20)
        .loc[:, ['timestamp', 'cmdb_id', 'message']]
        .copy()
    )
    # ensure timestamp column formatted
    error_sample_window['timestamp'] = pd.to_datetime(error_sample_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    error_counts_window = pd.DataFrame(columns=['cmdb_id','error_count','earliest_ts'])
    error_sample_window = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Final compact outputs (limited to 20 rows each)
metric_anom_top20, trace_anom_top20, log_anom_top20, error_counts_window, error_sample_window

```
Out[3]:
```
```python
# Plain-English summary of anomaly detection results (window: 2022-03-21 00:00:00 → 2022-03-21 00:30:00 Asia/Shanghai)
summary = (
    "Incident window: 2022-03-21 00:00:00 (Asia/Shanghai) inclusive → 2022-03-21 00:30:00 exclusive.\n"
    "Anomaly rule: a point is anomalous if value >= global P95 OR value <= global P5 (global thresholds computed from entire series).\n\n"
    "1) metric.csv anomalies (top findings, limited to 20 rows):\n"
    "- The anomalies are dominated by shippingservice2-0 container/node metrics (many 'container.node-5.*' KPIs).\n"
    "- For those shippingservice2-0 KPIs each series had anomaly_count = 30 (i.e., every point in the 30-minute window was flagged),\n"
    "  with earliest anomaly at 2022-03-21 00:00:00 Asia/Shanghai. Many of these flagged series are constant or zero-valued within the window\n"
    "  (example: container_memory_mapped_file, container_memory_max_usage_MB, container_memory_rss, container_memory_cache, and multiple fs/network counters).\n"
    "- Interpretation: these series hit their global P95 or P5 across the entire window, so every in-window sample was classified as anomalous.\n\n"
    "2) trace.csv anomalies (top findings, limited to 20 rows):\n"
    "- Many trace series flagged are error_rate metrics (e.g., trace.from_frontend-*.error_rate, trace.to_* .error_rate).\n"
    "- For the top trace entries listed, anomaly_count = 30 and earliest anomaly = 2022-03-21 00:00:00 Asia/Shanghai; in-window min/max values are 0.0.\n"
    "- Interpretation: error_rate series at 0.0 hit the global thresholds (P5/P95) and therefore were flagged for the whole window.\n\n"
    "3) log.csv anomalies (top findings, limited to 20 rows):\n"
    "- The log.error_count series across many components are flagged as anomalous (many entries show anomaly_count = 30, earliest 2022-03-21 00:00:00).\n"
    "- These flagged log.error_count values are 0.0 in-window; a few components had 28 points flagged instead of 30.\n"
    "- The system contains two log metrics (log.error_count and log.row_count); most reported anomalies are on log.error_count being at threshold.\n\n"
    "4) error_logs.csv within the same window:\n"
    "- Error counts by cmdb_id (top): frontend-0 = 229 rows (earliest 2022-03-21 00:00:11 CST), frontend-2 = 158 rows (earliest 00:00:05 CST), frontend-1 = 153 rows (earliest 00:00:12 CST).\n"
    "- Up to 20 earliest raw error log rows within the window were produced (they show repeated 'severity: warning, message: failed to retrieve...' across frontend-0/1/2).\n\n"
    "Overall takeaway:\n"
    "- The automated anomaly rule flagged many series across container/node KPIs and error_rate / log.error_count series for the entire window (30 points each).\n"
    "- Many of these flags appear to stem from series that are constant (including zeros) and match their global P5/P95 thresholds, resulting in every in-window sample being marked anomalous.\n"
    "- The most actionable signal is the high volume of frontend error log entries (frontend-0/1/2), which shows real logged warnings in the same window and should be investigated first.\n"
    "- Next steps: inspect shippingservice2-0 node/container KPIs (node-5 metrics) and frontend logs and traces to determine causal links; verify whether thresholding logic (global P5/P95) is appropriate for constant or zero-valued series.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  in_window_max_value  in_window_min_value
4824  shippingservice2-0      container.node-5.container_memory_mapped_file             30    2022-03-21 00:00:00 CST         4.915200e+05         4.915200e+05
4825  shippingservice2-0     container.node-5.container_memory_max_usage_MB             30    2022-03-21 00:00:00 CST         1.280117e+02         1.280117e+02
4827  shippingservice2-0             container.node-5.container_memory_swap             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4831  shippingservice2-0  container.node-5.container_network_receive_err...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4816  shippingservice2-0  container.node-5.container_fs_writes_merged./d...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4818  shippingservice2-0            container.node-5.container_memory_cache             30    2022-03-21 00:00:00 CST         1.913242e+07         1.913242e+07
4819  shippingservice2-0          container.node-5.container_memory_failcnt             30    2022-03-21 00:00:00 CST         2.397029e+06         2.397029e+06
4821  shippingservice2-0  container.node-5.container_memory_failures.con...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4823  shippingservice2-0  container.node-5.container_memory_failures.hie...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4808  shippingservice2-0  container.node-5.container_fs_reads_merged./de...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4809  shippingservice2-0  container.node-5.container_fs_sector_reads./de...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4810  shippingservice2-0  container.node-5.container_fs_sector_writes./d...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4812  shippingservice2-0  container.node-5.container_fs_write_seconds./d...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4813  shippingservice2-0      container.node-5.container_fs_writes./dev/vda             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4814  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4815  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4800  shippingservice2-0  container.node-5.container_fs_io_current./dev/...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4801  shippingservice2-0  container.node-5.container_fs_io_time_seconds....             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4802  shippingservice2-0  container.node-5.container_fs_io_time_weighted...             30    2022-03-21 00:00:00 CST         0.000000e+00         0.000000e+00
4803  shippingservice2-0   container.node-5.container_fs_limit_MB./dev/vda1             30    2022-03-21 00:00:00 CST         6.046307e+05         6.046307e+05,                        cmdb_id                                   trace_name  anomaly_count earliest_anomaly_timestamp  in_window_max_value  in_window_min_value
10                 adservice-0             trace.from_frontend-2.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
6                  adservice-0             trace.from_frontend-1.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
18                 adservice-1             trace.from_frontend-1.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
14                 adservice-1             trace.from_frontend-0.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
2                  adservice-0             trace.from_frontend-0.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
470          currencyservice-0             trace.from_frontend-2.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
462          currencyservice-0             trace.from_frontend-0.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
466          currencyservice-0             trace.from_frontend-1.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
486          currencyservice-1             trace.from_frontend-0.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
490          currencyservice-1             trace.from_frontend-1.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
1150  recommendationservice2-0  trace.to_productcatalogservice-2.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
1142  recommendationservice2-0  trace.to_productcatalogservice-0.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
1146  recommendationservice2-0  trace.to_productcatalogservice-1.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
1134  recommendationservice2-0            trace.from_frontend2-0.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
1138  recommendationservice2-0                        trace.self.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
1126   recommendationservice-2  trace.to_productcatalogservice-1.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
1130   recommendationservice-2  trace.to_productcatalogservice-2.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
86               cartservice-1             trace.from_frontend-1.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
90               cartservice-1             trace.from_frontend-2.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0
82               cartservice-1             trace.from_frontend-0.error_rate             30    2022-03-21 00:00:00 CST                  0.0                  0.0,                     cmdb_id         log_name  anomaly_count earliest_anomaly_timestamp  in_window_max_value  in_window_min_value
0               adservice-0  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
2               adservice-1  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
4               adservice-2  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
10            cartservice-2  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
8             cartservice-1  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
6             cartservice-0  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
18        currencyservice-0  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
22        currencyservice-2  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
20        currencyservice-1  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
50  recommendationservice-1  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
58        shippingservice-1  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
60        shippingservice-2  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
44  productcatalogservice-1  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
42  productcatalogservice-0  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
46  productcatalogservice-2  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
48  recommendationservice-0  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
52  recommendationservice-2  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
56        shippingservice-0  log.error_count             30    2022-03-21 00:00:00 CST                  0.0                  0.0
36         paymentservice-0  log.error_count             28    2022-03-21 00:00:00 CST                  0.0                  0.0
14        checkoutservice-1  log.error_count             28    2022-03-21 00:00:00 CST                  0.0                  0.0,       cmdb_id  error_count              earliest_ts
0  frontend-0          229  2022-03-21 00:00:11 CST
2  frontend-2          158  2022-03-21 00:00:05 CST
1  frontend-1          153  2022-03-21 00:00:12 CST,                   timestamp     cmdb_id                                            message
0   2022-03-21 00:00:05 CST  frontend-2  severity: warning, message: failed to retrieve...
1   2022-03-21 00:00:11 CST  frontend-0  severity: warning, message: failed to retrieve...
2   2022-03-21 00:00:12 CST  frontend-1  severity: warning, message: failed to retrieve...
3   2022-03-21 00:00:13 CST  frontend-2  severity: warning, message: failed to retrieve...
4   2022-03-21 00:00:13 CST  frontend-0  severity: warning, message: failed to retrieve...
5   2022-03-21 00:00:19 CST  frontend-1  severity: warning, message: failed to retrieve...
6   2022-03-21 00:00:21 CST  frontend-0  severity: warning, message: failed to retrieve...
7   2022-03-21 00:00:25 CST  frontend-1  severity: warning, message: failed to retrieve...
8   2022-03-21 00:00:26 CST  frontend-2  severity: warning, message: failed to retrieve...
9   2022-03-21 00:00:26 CST  frontend-0  severity: warning, message: failed to retrieve...
10  2022-03-21 00:00:30 CST  frontend-0  severity: warning, message: failed to retrieve...
11  2022-03-21 00:00:32 CST  frontend-1  severity: warning, message: failed to retrieve...
12  2022-03-21 00:00:36 CST  frontend-0  severity: warning, message: failed to retrieve...
13  2022-03-21 00:00:39 CST  frontend-1  severity: warning, message: failed to retrieve...
14  2022-03-21 00:00:39 CST  frontend-2  severity: warning, message: failed to retrieve...
15  2022-03-21 00:00:41 CST  frontend-0  severity: warning, message: failed to retrieve...
16  2022-03-21 00:00:45 CST  frontend-1  severity: warning, message: failed to retrieve...
17  2022-03-21 00:00:47 CST  frontend-0  severity: warning, message: failed to retrieve...
18  2022-03-21 00:00:50 CST  frontend-2  severity: warning, message: failed to retrieve...
19  2022-03-21 00:00:55 CST  frontend-0  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse previously computed variables where available:
# metric_anom_agg, trace_anom_agg, log_anom_agg, metric_thresh, trace_thresh, log_thresh

# Fallback to recompute minimal required vars if missing (shouldn't be needed in the current kernel state)
if 'metric_anom_agg' not in globals() or 'metric_thresh' not in globals():
    # minimal recompute using existing metric_win & metric_thresh if available
    metric_thresh = metric_thresh if 'metric_thresh' in globals() else (
        metric_df.dropna(subset=['value']).groupby(['cmdb_id','kpi_name'])['value']
                 .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)).reset_index()
    )
    metric_anom_agg = metric_anom_agg if 'metric_anom_agg' in globals() else aggregate_anomalies(metric_win, ['cmdb_id','kpi_name'])

if 'trace_anom_agg' not in globals() or 'trace_thresh' not in globals():
    trace_thresh = trace_thresh if 'trace_thresh' in globals() else (
        trace_df.dropna(subset=['value']).groupby(['cmdb_id','trace_name'])['value']
                .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)).reset_index()
    )
    trace_anom_agg = trace_anom_agg if 'trace_anom_agg' in globals() else aggregate_anomalies(trace_win, ['cmdb_id','trace_name'])

if 'log_anom_agg' not in globals() or 'log_thresh' not in globals():
    log_thresh = log_thresh if 'log_thresh' in globals() else (
        log_df.dropna(subset=['value']).groupby(['cmdb_id','log_name'])['value']
              .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)).reset_index()
    )
    log_anom_agg = log_anom_agg if 'log_anom_agg' in globals() else aggregate_anomalies(log_win, ['cmdb_id','log_name'])

# Helper to compute breach_factor per the provided formula
def compute_breach_factor(df_agg, thresh_df, name_col):
    # Merge global thresholds
    merged = df_agg.merge(thresh_df, left_on=['cmdb_id', name_col], right_on=['cmdb_id', name_col], how='left')
    # Ensure numeric columns
    for col in ['in_window_max_value', 'in_window_min_value', 'p95', 'p5']:
        if col in merged.columns:
            merged[col] = pd.to_numeric(merged[col], errors='coerce')
    # Compute breach factors safely
    def bf_row(r):
        max_ratio = 0.0
        min_ratio = 0.0
        # upper breach: in_window_max_value > global_p95 and global_p95>0
        if pd.notna(r.get('in_window_max_value')) and pd.notna(r.get('p95')) and r['p95'] > 0:
            if r['in_window_max_value'] > r['p95']:
                max_ratio = float(r['in_window_max_value']) / float(r['p95'])
        # lower breach: in_window_min_value < global_p5 and in_window_min_value>0
        if pd.notna(r.get('in_window_min_value')) and pd.notna(r.get('p5')) and r['in_window_min_value'] > 0:
            if r['in_window_min_value'] < r['p5']:
                min_ratio = float(r['p5']) / float(r['in_window_min_value'])
        return max(max_ratio, min_ratio)
    merged['breach_factor'] = merged.apply(bf_row, axis=1)
    # Keep only rows with breach_factor > 0
    merged_pos = merged[merged['breach_factor'] > 0].copy()
    # Select and order columns
    select_cols = ['cmdb_id', name_col, 'anomaly_count', 'earliest_anomaly_timestamp',
                   'in_window_min_value', 'in_window_max_value', 'p95', 'p5', 'breach_factor']
    # Rename p95/p5 to global_p95/global_p5 for clarity
    merged_pos = merged_pos.rename(columns={'p95': 'global_p95', 'p5': 'global_p5'})
    # Recompute select_cols with new names
    select_cols = ['cmdb_id', name_col, 'anomaly_count', 'earliest_anomaly_timestamp',
                   'in_window_min_value', 'in_window_max_value', 'global_p95', 'global_p5', 'breach_factor']
    # Sort by breach_factor desc and limit to top 20
    merged_pos = merged_pos.sort_values('breach_factor', ascending=False).reset_index(drop=True)
    return merged_pos.loc[:, select_cols].head(20), merged_pos  # return compact top20 and full merged table

# 1) Metric breaches
metric_breach_top20, metric_breach_full = compute_breach_factor(metric_anom_agg, metric_thresh, 'kpi_name')

# 2) Trace breaches
trace_breach_top20, trace_breach_full = compute_breach_factor(trace_anom_agg, trace_thresh, 'trace_name')

# 3) Log breaches
log_breach_top20, log_breach_full = compute_breach_factor(log_anom_agg, log_thresh, 'log_name')

# Final compact outputs (display variables; kernel will show them)
metric_breach_top20, trace_breach_top20, log_breach_top20

```
Out[4]:
```
```python
# Compact plain-English summary of ranked candidate faulty series by breach severity.
# Reuse existing kernel aggregates where available (metric_breach_top20, trace_breach_top20, log_breach_top20).

summary = (
    "Ranked candidate faulty series (based on breach_factor, window 2022-03-21 00:00→00:30 Asia/Shanghai):\n\n"
    "Metrics (top candidates):\n"
    " - checkoutservice-2 / container.node-5.container_cpu_usage_seconds: breach_factor ≈ 24.18, anomaly_count=4, earliest=2022-03-21 00:00:00. (High CPU breach relative to global P95)\n"
    " - paymentservice-2, emailservice-0, emailservice-2, emailservice2-0, shippingservice-2, adservice-*, cartservice-* series: large breach_factors (≈22→12) on container.node-5.container_network_receive_MB... (network receive spiking above global P95).\n"
    " - node-4 / system.net.tcp.retrans_segs: breach_factor ≈ 17.67, anomaly_count=21 — elevated TCP retransmissions.\n"
    " - node-3 / system.io.avg_q_sz: breach_factor ≈ 9.09, anomaly_count=30 — high IO queue size observed.\n\n"
    "Traces (top candidates):\n"
    " - cartservice-0 traces to/from checkoutservice: multiple trace.series (duration_mean and duration_p95) show breach_factor = 20 (many anomalous points). These indicate latency spikes in cart↔checkout interactions.\n"
    " - Several frontend→adservice and adservice from_frontend duration_p95 entries (breach_factors ≈ 9.7→4.6) — elevated downstream latencies on some call paths.\n\n"
    "Logs (top candidates):\n"
    " - checkoutservice-2 / log.row_count and checkoutservice-1 / log.row_count: breach_factor ≈ 4.67 (anomaly_count 6 and 10) — large deviations in row_count.\n"
    " - Several services (shippingservice-0, emailservice-0, paymentservice-*) show row_count deviations (breach_factor ~1.2–2.3).\n"
    " - frontend error_count entries appear in the list but with lower breach_factor (~1.2–1.3).\n\n"
    "Overall assessment and recommended next steps:\n"
    " 1) Prioritize investigation of the highest breach_factor metric series: checkoutservice-2 CPU and the multiple container.network_receive spikes (payment/email/shipping/ad/cart services). These represent the largest relative breaches.\n"
    " 2) Correlate with trace findings: cart↔checkout and frontend→adservice trace anomalies point to request latency impact along those paths — focus on checkout/cart interactions.\n"
    " 3) Check node-level signals: node-4 TCP retransmissions and node-3 IO queue size are notable and may indicate network or disk issues affecting containers.\n"
    " 4) Review logs for checkoutservice and frontend error rows (high error_count and many error_log entries for frontend-0/1/2) for contextual error messages.\n\n"
    "Conclusion: top candidates are a mix of container network receive spikes across multiple services and a checkoutservice CPU breach; investigate these metrics, node-level retransmissions/IO, and the cart↔checkout trace paths in parallel, while using frontend error logs as corroborating evidence."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  in_window_min_value  in_window_max_value  global_p95  global_p5  breach_factor
0         checkoutservice-2       container.node-5.container_cpu_usage_seconds              4    2022-03-21 00:00:00 CST             0.000233             0.075101    0.053704   0.005644      24.182514
1          paymentservice-2  container.node-5.container_network_receive_MB....              4    2022-03-21 00:03:00 CST             0.017980             0.548554    0.024183   0.018089      22.683373
2            emailservice-0  container.node-5.container_network_receive_MB....              5    2022-03-21 00:00:00 CST             0.018024             0.551865    0.025436   0.018247      21.696144
3            emailservice-2  container.node-5.container_network_receive_MB....              4    2022-03-21 00:14:00 CST             0.018147             0.551337    0.025484   0.018981      21.634471
4           emailservice2-0  container.node-5.container_network_receive_MB....              4    2022-03-21 00:05:00 CST             0.019135             0.552085    0.025721   0.019668      21.464203
5         shippingservice-2  container.node-5.container_network_receive_MB....              4    2022-03-21 00:15:00 CST             0.020908             0.555237    0.030760   0.023142      18.050834
6                    node-4                        system.net.tcp.retrans_segs             21    2022-03-21 00:00:00 CST             0.000000             0.530000    0.030000   0.000000      17.666667
7         shippingservice-0  container.node-5.container_network_receive_MB....              4    2022-03-21 00:06:00 CST             0.019578             0.564911    0.037861   0.022460      14.920668
8               adservice-1  container.node-5.container_network_receive_MB....              5    2022-03-21 00:00:00 CST             0.025125             0.575804    0.041861   0.027359      13.755274
9               adservice-0  container.node-5.container_network_receive_MB....              4    2022-03-21 00:05:00 CST             0.034698             0.582744    0.045894   0.035720      12.697632
10            cartservice-2  container.node-5.container_network_receive_MB....              4    2022-03-21 00:04:00 CST             0.030486             0.567611    0.045080   0.034312      12.591209
11            cartservice-0  container.node-5.container_network_receive_MB....              4    2022-03-21 00:15:00 CST             0.032064             0.584590    0.046580   0.032413      12.550221
12             adservice2-0  container.node-6.container_network_receive_MB....              4    2022-03-21 00:04:00 CST             0.030783             0.581724    0.058208   0.033048       9.993894
13                   node-3                                 system.io.avg_q_sz             30    2022-03-21 00:00:00 CST             0.000000             0.050000    0.005500   0.000000       9.090909
14        currencyservice-2  container.node-5.container_network_receive_MB....              4    2022-03-21 00:02:00 CST             0.047317             0.596007    0.070243   0.048736       8.484880
15        currencyservice-0  container.node-5.container_network_receive_MB....              4    2022-03-21 00:00:00 CST             0.046625             0.585999    0.071611   0.047118       8.183078
16                   node-1                        system.net.tcp.retrans_segs              4    2022-03-21 00:10:00 CST             0.390000             7.670000    1.041500   0.404500       7.364378
17  recommendationservice-2  container.node-5.container_network_receive_MB....              4    2022-03-21 00:05:00 CST             0.056109             0.597071    0.082215   0.059596       7.262319
18  recommendationservice-1  container.node-5.container_network_receive_MB....              5    2022-03-21 00:00:00 CST             0.057162             0.609010    0.084219   0.058175       7.231306
19               frontend-1  mesh.source.frontend.shippingservice.istio_res...              4    2022-03-21 00:23:00 CST             0.500000             9.000000    8.275000   2.450000       4.900000,                     cmdb_id                                       trace_name  anomaly_count earliest_anomaly_timestamp  in_window_min_value  in_window_max_value  global_p95  global_p5  breach_factor
0             cartservice-0       trace.from_checkoutservice-1.duration_mean             20    2022-03-21 00:00:00 CST             0.000000             0.001000    0.000050   0.000000      20.000000
1             cartservice-0        trace.from_checkoutservice-1.duration_p95             20    2022-03-21 00:00:00 CST             0.000000             0.001000    0.000050   0.000000      20.000000
2         checkoutservice-1              trace.to_cartservice-0.duration_p95             20    2022-03-21 00:00:00 CST             0.000000             0.001000    0.000050   0.000000      20.000000
3         checkoutservice-1             trace.to_cartservice-0.duration_mean             20    2022-03-21 00:00:00 CST             0.000000             0.001000    0.000050   0.000000      20.000000
4                frontend-1                trace.to_adservice-2.duration_p95              9    2022-03-21 00:03:00 CST             0.000012             0.000186    0.000019   0.000013       9.738220
5               adservice-2               trace.from_frontend-1.duration_p95              9    2022-03-21 00:03:00 CST             0.000012             0.000186    0.000019   0.000013       9.738220
6               adservice-2               trace.from_frontend-2.duration_p95              9    2022-03-21 00:05:00 CST             0.000013             0.000140    0.000024   0.000013       5.919662
7                frontend-2                trace.to_adservice-2.duration_p95              9    2022-03-21 00:05:00 CST             0.000013             0.000140    0.000024   0.000013       5.919662
8         checkoutservice-2         trace.to_currencyservice-0.duration_mean              5    2022-03-21 00:01:00 CST             0.000091             0.000681    0.000117   0.000093       5.810580
9         currencyservice-0       trace.from_checkoutservice-2.duration_mean              5    2022-03-21 00:01:00 CST             0.000091             0.000681    0.000117   0.000093       5.810580
10        currencyservice-0        trace.from_checkoutservice-2.duration_p95              4    2022-03-21 00:01:00 CST             0.000093             0.000681    0.000119   0.000093       5.703518
11        checkoutservice-2          trace.to_currencyservice-0.duration_p95              4    2022-03-21 00:01:00 CST             0.000093             0.000681    0.000119   0.000093       5.703518
12              adservice-2              trace.from_frontend-1.duration_mean              7    2022-03-21 00:03:00 CST             0.000012             0.000077    0.000015   0.000012       5.292096
13               frontend-1               trace.to_adservice-2.duration_mean              7    2022-03-21 00:03:00 CST             0.000012             0.000077    0.000015   0.000012       5.292096
14  recommendationservice-1    trace.to_productcatalogservice-2.duration_p95              6    2022-03-21 00:00:00 CST             0.000023             0.000191    0.000038   0.000024       5.026316
15  productcatalogservice-2  trace.from_recommendationservice-1.duration_p95              6    2022-03-21 00:00:00 CST             0.000023             0.000191    0.000038   0.000024       5.026316
16               frontend-0                trace.to_adservice-1.duration_p95             16    2022-03-21 00:01:00 CST             0.000013             0.000148    0.000032   0.000013       4.589147
17              adservice-1               trace.from_frontend-0.duration_p95             16    2022-03-21 00:01:00 CST             0.000013             0.000148    0.000032   0.000013       4.589147
18  productcatalogservice-0  trace.from_recommendationservice-2.duration_p95              4    2022-03-21 00:00:00 CST             0.000023             0.000192    0.000045   0.000024       4.295302
19  recommendationservice-2    trace.to_productcatalogservice-0.duration_p95              4    2022-03-21 00:00:00 CST             0.000023             0.000192    0.000045   0.000024       4.295302,               cmdb_id         log_name  anomaly_count earliest_anomaly_timestamp  in_window_min_value  in_window_max_value  global_p95  global_p5  breach_factor
0   checkoutservice-2    log.row_count              6    2022-03-21 00:01:00 CST                  3.0                 48.0       34.00      14.00       4.666667
1   checkoutservice-1    log.row_count             10    2022-03-21 00:02:00 CST                  3.0                 34.0       34.00      14.00       4.666667
2   shippingservice-0    log.row_count              4    2022-03-21 00:00:00 CST                  6.0                 53.0       49.10      13.80       2.300000
3      emailservice-0    log.row_count              4    2022-03-21 00:03:00 CST                  1.0                  8.0        7.60       2.20       2.200000
4    paymentservice-2    log.row_count             18    2022-03-21 00:00:00 CST                  2.0                  9.0        8.40       3.00       1.500000
5    paymentservice-1    log.row_count              4    2022-03-21 00:04:00 CST                  1.0                  9.0        7.60       1.40       1.400000
6   shippingservice-2    log.row_count              6    2022-03-21 00:00:00 CST                 13.0                 51.0       45.55      18.00       1.384615
7    paymentservice-0    log.row_count              8    2022-03-21 00:02:00 CST                  1.0                  6.0        6.00       1.35       1.350000
8        redis-cart-0    log.row_count              5    2022-03-21 00:00:00 CST                  2.0                  5.0        5.00       2.60       1.300000
9          frontend-1  log.error_count             10    2022-03-21 00:00:00 CST                  3.0                  9.0        7.00       3.00       1.285714
10     emailservice-2    log.row_count              6    2022-03-21 00:01:00 CST                  1.0                  9.0        7.00       1.00       1.285714
11         frontend-2  log.error_count              6    2022-03-21 00:06:00 CST                  2.0                  8.0        8.00       2.45       1.225000
12     emailservice-1    log.row_count              5    2022-03-21 00:02:00 CST                  1.0                  9.0        8.00       1.20       1.200000
13         frontend-0  log.error_count             10    2022-03-21 00:00:00 CST                  5.0                 10.0       10.00       6.00       1.200000
14         frontend-0    log.row_count              4    2022-03-21 00:03:00 CST                440.0                750.0      711.15     509.25       1.157386
15        adservice-2    log.row_count              5    2022-03-21 00:00:00 CST                 26.0                 46.0       44.20      30.00       1.153846
16  currencyservice-2    log.row_count              4    2022-03-21 00:00:00 CST                192.0                322.0      282.75     217.90       1.138815
17  currencyservice-0    log.row_count              4    2022-03-21 00:00:00 CST                202.0                321.0      282.65     215.25       1.135680
18  currencyservice-1    log.row_count              4    2022-03-21 00:00:00 CST                202.0                320.0      282.30     216.60       1.133546
19      cartservice-0    log.row_count              5    2022-03-21 00:01:00 CST                220.0                329.0      328.00     246.05       1.118409)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse variables from kernel:
# metric_breach_top20 (compact top20), metric_breach_full (full merged), metric_win (in-window full metric rows), metric_thresh (global p95/p5)
# If some are missing, attempt minimal recompute (but prefer reuse)
if 'metric_breach_top20' not in globals():
    # fallback: compute metric_breach_top20 from metric_breach_full if available
    if 'metric_breach_full' in globals():
        metric_breach_top20 = metric_breach_full.sort_values('breach_factor', ascending=False).head(20)
    else:
        raise RuntimeError("Required variables metric_breach_top20 or metric_breach_full not found in kernel.")

# Ensure metric_win and metric_thresh exist
if 'metric_win' not in globals():
    # load/prepare metric_win from metric_df using previously used window
    metric_df = metric_df if 'metric_df' in globals() else pd.read_csv("metric.csv")
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    if '_ts_dt' not in metric_df.columns:
        metric_df['_ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_dt = pd.Timestamp("2022-03-21 00:00:00").tz_localize(tz)
    end_dt = pd.Timestamp("2022-03-21 00:30:00").tz_localize(tz)
    metric_win = metric_df[(metric_df['_ts_dt'] >= start_dt) & (metric_df['_ts_dt'] < end_dt)].copy()

if 'metric_thresh' not in globals():
    metric_thresh = (
        metric_df.dropna(subset=['value'])
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
    )

# Select top 15 candidate series by breach_factor from metric_breach_top20
top_candidates = metric_breach_top20.sort_values('breach_factor', ascending=False).head(15).copy()

# Prepare containers for storing full per-series segment tables and summary rows
series_segments = {}   # store full segments per series (DataFrame)
summary_rows = []

# Helper to compute consecutive anomalous segments for a single series DataFrame
def find_consecutive_segments(df):
    # df must be sorted by _ts_dt ascending and contain columns: _ts_dt, value, p95, p5, is_anom
    anom_df = df[df['is_anom']].copy()
    if anom_df.empty:
        return pd.DataFrame(columns=['segment_id','start_ts','end_ts','count','max_value','min_value'])
    anom_df = anom_df.sort_values('_ts_dt').reset_index(drop=True)
    # compute time difference in seconds between consecutive anomaly points
    anom_df['_dt_diff'] = anom_df['_ts_dt'].diff().fillna(pd.Timedelta(seconds=0))
    # start new segment when diff != 1 minute
    anom_df['_new_seg'] = (anom_df['_dt_diff'] != pd.Timedelta(minutes=1))
    anom_df['_seg_id'] = anom_df['_new_seg'].cumsum()
    segs = (
        anom_df.groupby('_seg_id')
        .agg(start_ts=('_ts_dt','min'),
             end_ts=('_ts_dt','max'),
             count=('value','size'),
             max_value=('value','max'),
             min_value=('value','min'))
        .reset_index(drop=True)
    )
    return segs

# Iterate candidates
for _, row in top_candidates.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    # filter metric_win for this series
    ser = metric_win[(metric_win['cmdb_id'] == cmdb) & (metric_win['kpi_name'] == kpi)].copy()
    if ser.empty:
        continue
    # merge global thresholds for safety
    thr = metric_thresh[(metric_thresh['cmdb_id'] == cmdb) & (metric_thresh['kpi_name'] == kpi)]
    if thr.empty:
        # no thresholds, skip
        continue
    global_p95 = float(thr['p95'].iloc[0])
    global_p5 = float(thr['p5'].iloc[0])
    ser['p95'] = global_p95
    ser['p5'] = global_p5
    # mark anomalies per rule
    ser['is_anom'] = ser.apply(lambda r: False if pd.isna(r['value']) else (r['value'] >= r['p95'] or r['value'] <= r['p5']), axis=1)
    # find consecutive anomalous segments
    segs = find_consecutive_segments(ser)
    # store full segments DataFrame for this series
    series_key = f"{cmdb}||{kpi}"
    series_segments[series_key] = segs.copy()
    if segs.empty:
        continue
    # take earliest anomalous segment (first row)
    first = segs.iloc[0]
    start_ts = first['start_ts']
    end_ts = first['end_ts']
    duration_min = int((end_ts - start_ts) / pd.Timedelta(minutes=1)) + 1  # inclusive minutes
    anomaly_count = int(first['count'])
    in_seg_max = float(first['max_value'])
    in_seg_min = float(first['min_value'])
    # compute breach_ratio per rule
    breach_ratio = 0.0
    if (in_seg_max > global_p95) and (global_p95 > 0):
        breach_ratio = in_seg_max / global_p95
    elif (in_seg_min < global_p5) and (in_seg_min > 0):
        breach_ratio = global_p5 / in_seg_min
    else:
        breach_ratio = 0.0
    # only include if breach_ratio>0 per instructions
    if breach_ratio > 0:
        summary_rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'earliest_anomaly_timestamp': start_ts.strftime('%Y-%m-%d %H:%M:%S %Z'),
            'latest_anomaly_timestamp': end_ts.strftime('%Y-%m-%d %H:%M:%S %Z'),
            'duration_minutes': duration_min,
            'anomaly_count_in_segment': anomaly_count,
            'in_segment_max_value': in_seg_max,
            'in_segment_min_value': in_seg_min,
            'global_p95': global_p95,
            'global_p5': global_p5,
            'breach_ratio': breach_ratio
        })

# Create summary DataFrame, sort by breach_ratio desc, limit to 15 rows
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values('breach_ratio', ascending=False).reset_index(drop=True).head(15)

# Keep full per-series segments in kernel variable 'metric_series_segments_full'
metric_series_segments_full = series_segments

# Display the compact summary table
summary_df

```
Out[5]:
```
```python
summary = (
    "Top metric anomaly segments (window 2022-03-21 00:00→00:30 Asia/Shanghai):\n\n"
    "1) Highest severity single-minute network spikes:\n"
    "   - paymentservice-2 / container.node-5.container_network_receive...: breach_ratio ≈ 22.68 (1 anomalous point at 2022-03-21 00:03), in-segment value=0.5486 vs global_p95=0.02418.\n"
    "   - emailservice2-0 / container.node-5.container_network_receive...: breach_ratio ≈ 21.46 (1 point at 00:05), in-segment value=0.5521 vs global_p95=0.02572.\n"
    "   - adservice-0 / container.node-5.container_network_receive...: breach_ratio ≈ 12.70 (2-minute segment 00:05–00:06), in-seg max=0.5827 vs global_p95=0.04589.\n\n"
    "2) Sustained node-level I/O issue:\n"
    "   - node-3 / system.io.avg_q_sz: breach_ratio ≈ 9.09, anomalous for entire 30-minute window (00:00–00:29), anomaly_count=30, in-seg max=0.05 vs global_p95=0.0055 — indicates sustained high IO queue depth.\n\n"
    "3) Other notable items:\n"
    "   - checkoutservice-2 / container.node-5.container_cpu_usage_seconds: breach_ratio ≈ 1.40, 3-minute segment (00:00–00:02), suggests a short CPU elevation.\n"
    "   - Multiple other services (shippingservice/payment/email/ad/cart/currency/recommendation) show smaller yet >1 breach_ratios on container.node-5.network_receive series (single-minute spikes or short segments).\n\n"
    "Observations and recommended next steps:\n"
    " - Prioritize investigation of the high-breach network spikes (paymentservice-2, emailservice2-0, adservice-0, etc.) on node-5 — these are the largest relative deviations.\n"
    " - Investigate node-3 I/O subsystem immediately due to a sustained 30-minute elevated avg_q_sz.\n"
    " - Check checkoutservice-2 CPU segment and correlate with traces/logs for that 00:00–00:02 interval.\n"
    " - Correlate these metric segments with trace and log anomalies (cart↔checkout traces and frontend error logs) to find causal paths.\n\n"
    "Note: many metric breaches are single-minute spikes; node-3 I/O is the only sustained 30-min segment in this top list. "
    "Use the per-series segment tables stored in kernel variable 'metric_series_segments_full' for detailed timestamps and values."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name earliest_anomaly_timestamp latest_anomaly_timestamp  duration_minutes  anomaly_count_in_segment  in_segment_max_value  in_segment_min_value  global_p95  global_p5  breach_ratio
0    paymentservice-2  container.node-5.container_network_receive_MB....    2022-03-21 00:03:00 CST  2022-03-21 00:03:00 CST                 1                         1              0.548554              0.548554    0.024183   0.018089     22.683373
1     emailservice2-0  container.node-5.container_network_receive_MB....    2022-03-21 00:05:00 CST  2022-03-21 00:05:00 CST                 1                         1              0.552085              0.552085    0.025721   0.019668     21.464203
2         adservice-0  container.node-5.container_network_receive_MB....    2022-03-21 00:05:00 CST  2022-03-21 00:06:00 CST                 2                         2              0.582744              0.035371    0.045894   0.035720     12.697632
3              node-3                                 system.io.avg_q_sz    2022-03-21 00:00:00 CST  2022-03-21 00:29:00 CST                30                        30              0.050000              0.000000    0.005500   0.000000      9.090909
4   checkoutservice-2       container.node-5.container_cpu_usage_seconds    2022-03-21 00:00:00 CST  2022-03-21 00:02:00 CST                 3                         3              0.075101              0.000233    0.053704   0.005644      1.398427
5   shippingservice-2  container.node-5.container_network_receive_MB....    2022-03-21 00:15:00 CST  2022-03-21 00:15:00 CST                 1                         1              0.020908              0.020908    0.030760   0.023142      1.106851
6         adservice-1  container.node-5.container_network_receive_MB....    2022-03-21 00:00:00 CST  2022-03-21 00:02:00 CST                 3                         3              0.041861              0.025125    0.041861   0.027359      1.088946
7      emailservice-2  container.node-5.container_network_receive_MB....    2022-03-21 00:14:00 CST  2022-03-21 00:14:00 CST                 1                         1              0.018147              0.018147    0.025484   0.018981      1.045985
8        adservice2-0  container.node-6.container_network_receive_MB....    2022-03-21 00:04:00 CST  2022-03-21 00:04:00 CST                 1                         1              0.060020              0.060020    0.058208   0.033048      1.031138
9   currencyservice-2  container.node-5.container_network_receive_MB....    2022-03-21 00:02:00 CST  2022-03-21 00:02:00 CST                 1                         1              0.072428              0.072428    0.070243   0.048736      1.031104
10      cartservice-2  container.node-5.container_network_receive_MB....    2022-03-21 00:04:00 CST  2022-03-21 00:04:00 CST                 1                         1              0.046011              0.046011    0.045080   0.034312      1.020653
11  shippingservice-0  container.node-5.container_network_receive_MB....    2022-03-21 00:06:00 CST  2022-03-21 00:06:00 CST                 1                         1              0.022132              0.022132    0.037861   0.022460      1.014795
12      cartservice-0  container.node-5.container_network_receive_MB....    2022-03-21 00:15:00 CST  2022-03-21 00:17:00 CST                 3                         3              0.047206              0.032064    0.046580   0.032413      1.013447```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing trace breach table if available
if 'trace_breach_top20' in globals():
    trace_df = trace_breach_top20.copy()
elif 'trace_breach_full' in globals():
    trace_df = trace_breach_full.copy()
else:
    # fallback: if trace_anom_agg and trace_thresh exist, join them to recreate similar table
    if 'trace_anom_agg' in globals() and 'trace_thresh' in globals():
        trace_df = trace_anom_agg.merge(trace_thresh, on=['cmdb_id','trace_name'], how='left')
        trace_df = trace_df.rename(columns={'p95':'global_p95','p5':'global_p5'})
        trace_df['breach_factor'] = trace_df.apply(
            lambda r: max(
                (r['in_window_max_value'] / r['global_p95']) if (pd.notna(r.get('in_window_max_value')) and pd.notna(r.get('global_p95')) and r['in_window_max_value'] > r['global_p95'] and r['global_p95']>0) else 0,
                (r['global_p5'] / r['in_window_min_value']) if (pd.notna(r.get('in_window_min_value')) and pd.notna(r.get('global_p5')) and r['in_window_min_value'] < r['global_p5'] and r['in_window_min_value']>0) else 0
            ), axis=1
        )
    else:
        raise RuntimeError("Required trace breach table not found in kernel and cannot be reconstructed.")

# Ensure columns exist
expected_cols = ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','in_window_max_value','in_window_min_value','breach_factor']
for c in expected_cols:
    if c not in trace_df.columns:
        # If missing some columns, try to source from trace_anom_agg where names differ
        pass

# Parse earliest_anomaly_timestamp robustly: strip timezone token and localize to Asia/Shanghai
trace_df = trace_df.copy()
trace_df['earliest_ts_str'] = trace_df['earliest_anomaly_timestamp'].astype(str)
# Remove trailing timezone token if present (e.g., "2022-03-21 00:00:00 CST")
trace_df['earliest_ts_naive_str'] = trace_df['earliest_ts_str'].str.rsplit(' ', n=1).str[0]
trace_df['earliest_ts_dt'] = pd.to_datetime(trace_df['earliest_ts_naive_str'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
trace_df['earliest_ts_dt'] = trace_df['earliest_ts_dt'].dt.tz_localize(tz)

# Define windows
early_cut = pd.Timestamp("2022-03-21 00:01:00").tz_localize(tz)  # <= this for Failure A
late_start = pd.Timestamp("2022-03-21 00:02:00").tz_localize(tz)  # inclusive
late_end = pd.Timestamp("2022-03-21 00:10:00").tz_localize(tz)    # exclusive

# Tag if trace_name contains any of keywords
keywords = ['checkout','cart','frontend','currency','adservice']
trace_df['tag_callchain'] = trace_df['trace_name'].str.lower().apply(lambda s: any(kw in s for kw in keywords))

# 1) Early-window traces: earliest_anomaly_timestamp <= early_cut
trace_early = trace_df[trace_df['earliest_ts_dt'].notna() & (trace_df['earliest_ts_dt'] <= early_cut)].copy()
trace_early = trace_early.sort_values('breach_factor', ascending=False).head(20)

trace_early_candidates = trace_early.loc[:, ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
                                             'in_window_max_value','in_window_min_value','breach_factor','tag_callchain']].reset_index(drop=True)

# 2) Later-window traces: earliest between late_start (inclusive) and late_end (exclusive)
trace_late = trace_df[trace_df['earliest_ts_dt'].notna() & (trace_df['earliest_ts_dt'] >= late_start) & (trace_df['earliest_ts_dt'] < late_end)].copy()
trace_late = trace_late.sort_values('breach_factor', ascending=False).head(20)

trace_late_candidates = trace_late.loc[:, ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
                                           'in_window_max_value','in_window_min_value','breach_factor','tag_callchain']].reset_index(drop=True)

# Store results in kernel variables for follow-up steps
trace_early_candidates, trace_late_candidates

```
Out[6]:
```
```python
# Plain-English summary of trace-based localization (early-window = Failure A, later-window = Failure B).
summary = (
    "Trace localization summary:\n\n"
    "Failure A — early-window (earliest_anomaly_timestamp <= 2022-03-21 00:01:00 Asia/Shanghai):\n"
    "- Strong signals at 00:00–00:01 from cartservice ⇄ checkoutservice traces (multiple series, breach_factor = 20).\n"
    "- Related traces implicate checkoutservice → currencyservice and currencyservice ← checkoutservice around 00:01 (breach_factors ~5.7–5.8).\n"
    "- Several frontend→adservice and adservice-from-frontend p95 traces at ~00:01 also flagged (breach_factors ~4.6).\n"
    "- Tagging shows key call-chain keywords (checkout, cart, frontend, currency, adservice) are present in many early hits — primary suspect: cart↔checkout path (with early currency involvement).\n\n"
    "Failure B — later-window (2022-03-21 00:02:00 ≤ earliest < 00:10:00 Asia/Shanghai):\n"
    "- Clear spike at ~00:03 in frontend→adservice and adservice-from-frontend p95 traces (breach_factor ≈ 9.7).\n"
    "- Additional adservice duration anomalies at ~00:05 (breach_factor ~5.9).\n"
    "- Tagging shows these are in the frontend↔adservice call path — primary suspect for Failure B: adservice latency impacting frontend requests.\n\n"
    "Overall interpretation and recommended next steps:\n"
    "1) Treat this as two candidate failures:\n"
    "   - Failure A (early): cart/checkout interaction (00:00–00:01) possibly propagating to currency calls.\n"
    "   - Failure B (slightly later): frontend→adservice latency cluster (00:03–00:05).\n"
    "2) Correlate these trace groups with metrics and logs: check cart/checkout service metrics and currency traces for Failure A; check adservice metrics, frontend traces, and network metrics for Failure B.\n"
    "3) Use frontend error logs and the previously reported metric anomalies (node IO and container network spikes) to prioritize investigation and determine whether node/network issues caused the trace degradations.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                       trace_name  anomaly_count earliest_anomaly_timestamp  in_window_max_value  in_window_min_value  breach_factor  tag_callchain
0             cartservice-0       trace.from_checkoutservice-1.duration_mean             20    2022-03-21 00:00:00 CST             0.001000             0.000000      20.000000           True
1             cartservice-0        trace.from_checkoutservice-1.duration_p95             20    2022-03-21 00:00:00 CST             0.001000             0.000000      20.000000           True
2         checkoutservice-1              trace.to_cartservice-0.duration_p95             20    2022-03-21 00:00:00 CST             0.001000             0.000000      20.000000           True
3         checkoutservice-1             trace.to_cartservice-0.duration_mean             20    2022-03-21 00:00:00 CST             0.001000             0.000000      20.000000           True
4         checkoutservice-2         trace.to_currencyservice-0.duration_mean              5    2022-03-21 00:01:00 CST             0.000681             0.000091       5.810580           True
5         currencyservice-0       trace.from_checkoutservice-2.duration_mean              5    2022-03-21 00:01:00 CST             0.000681             0.000091       5.810580           True
6         currencyservice-0        trace.from_checkoutservice-2.duration_p95              4    2022-03-21 00:01:00 CST             0.000681             0.000093       5.703518           True
7         checkoutservice-2          trace.to_currencyservice-0.duration_p95              4    2022-03-21 00:01:00 CST             0.000681             0.000093       5.703518           True
8   recommendationservice-1    trace.to_productcatalogservice-2.duration_p95              6    2022-03-21 00:00:00 CST             0.000191             0.000023       5.026316          False
9   productcatalogservice-2  trace.from_recommendationservice-1.duration_p95              6    2022-03-21 00:00:00 CST             0.000191             0.000023       5.026316          False
10               frontend-0                trace.to_adservice-1.duration_p95             16    2022-03-21 00:01:00 CST             0.000148             0.000013       4.589147           True
11              adservice-1               trace.from_frontend-0.duration_p95             16    2022-03-21 00:01:00 CST             0.000148             0.000013       4.589147           True
12  productcatalogservice-0  trace.from_recommendationservice-2.duration_p95              4    2022-03-21 00:00:00 CST             0.000192             0.000023       4.295302          False
13  recommendationservice-2    trace.to_productcatalogservice-0.duration_p95              4    2022-03-21 00:00:00 CST             0.000192             0.000023       4.295302          False,        cmdb_id                           trace_name  anomaly_count earliest_anomaly_timestamp  in_window_max_value  in_window_min_value  breach_factor  tag_callchain
0   frontend-1    trace.to_adservice-2.duration_p95              9    2022-03-21 00:03:00 CST             0.000186             0.000012       9.738220           True
1  adservice-2   trace.from_frontend-1.duration_p95              9    2022-03-21 00:03:00 CST             0.000186             0.000012       9.738220           True
2  adservice-2   trace.from_frontend-2.duration_p95              9    2022-03-21 00:05:00 CST             0.000140             0.000013       5.919662           True
3   frontend-2    trace.to_adservice-2.duration_p95              9    2022-03-21 00:05:00 CST             0.000140             0.000013       5.919662           True
4  adservice-2  trace.from_frontend-1.duration_mean              7    2022-03-21 00:03:00 CST             0.000077             0.000012       5.292096           True
5   frontend-1   trace.to_adservice-2.duration_mean              7    2022-03-21 00:03:00 CST             0.000077             0.000012       5.292096           True)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables from kernel where available
# metric_win (in-window metric rows with is_anom), metric_anom_agg (aggregated anomalies), metric_thresh,
# metric_breach_full (full merged with breach_factor), trace_anom_agg (aggregated trace anomalies)

# Fallback safeguards: if variables missing, try to recreate minimal needed data
if 'metric_win' not in globals():
    metric_df = metric_df if 'metric_df' in globals() else pd.read_csv("metric.csv")
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    if '_ts_dt' not in metric_df.columns:
        metric_df['_ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_dt = pd.Timestamp("2022-03-21 00:00:00").tz_localize(tz)
    end_dt = pd.Timestamp("2022-03-21 00:30:00").tz_localize(tz)
    metric_win = metric_df[(metric_df['_ts_dt'] >= start_dt) & (metric_df['_ts_dt'] < end_dt)].copy()

if 'metric_anom_agg' not in globals():
    # try to reconstruct from metric_win if is_anom exists, else we cannot proceed
    if 'is_anom' not in metric_win.columns:
        # need thresholds to mark anomalies
        if 'metric_thresh' not in globals():
            metric_thresh = (
                metric_df.dropna(subset=['value'])
                .groupby(['cmdb_id','kpi_name'])['value']
                .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
                .reset_index()
            )
        metric_win = metric_win.merge(metric_thresh, on=['cmdb_id','kpi_name'], how='left')
        metric_win['is_anom'] = metric_win.apply(
            lambda r: False if pd.isna(r['value']) or pd.isna(r['p95']) or pd.isna(r['p5']) else (r['value'] >= r['p95'] or r['value'] <= r['p5']),
            axis=1
        )
    metric_anom_agg = (
        metric_win[metric_win['is_anom']]
        .groupby(['cmdb_id','kpi_name'])
        .agg(anomaly_count=('is_anom','sum'),
             earliest_anomaly_timestamp=('_ts_dt','min'),
             in_window_max_value=('value','max'),
             in_window_min_value=('value','min'))
        .reset_index()
    )
    metric_anom_agg['earliest_anomaly_timestamp'] = metric_anom_agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

if 'trace_anom_agg' not in globals() and 'trace_win' in globals():
    trace_anom_agg = (
        trace_win[trace_win['is_anom']]
        .groupby(['cmdb_id','trace_name'])
        .agg(anomaly_count=('is_anom','sum'),
             earliest_anomaly_timestamp=('_ts_dt','min'),
             in_window_max_value=('value','max'),
             in_window_min_value=('value','min'))
        .reset_index()
    )
    trace_anom_agg['earliest_anomaly_timestamp'] = trace_anom_agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# --- 1) Node-to-service anomaly table for node-3 and node-5 ---
node_tags = ['node-3', 'node-5']
node_rows = []

for node_tag in node_tags:
    # find kpis containing node_tag and flagged as anomalous in-window
    mask = metric_win['kpi_name'].astype(str).str.contains(node_tag, na=False) & (metric_win['is_anom'] == True)
    flagged = metric_win[mask].copy()
    if flagged.empty:
        continue
    # group by cmdb_id
    grp = flagged.groupby('cmdb_id').agg(
        number_of_distinct_flagged_kpis=('kpi_name', lambda s: s.dropna().unique().size),
        total_anomalous_points_in_window=('is_anom','sum'),
        earliest_anomaly_ts=('_ts_dt','min')
    ).reset_index()
    # collect example kpis up to 5 for each cmdb_id
    example_kpis = (
        flagged.groupby('cmdb_id')['kpi_name']
        .apply(lambda s: list(pd.Index(s.dropna().unique())[:5]))
        .reset_index()
        .rename(columns={'kpi_name':'example_kpis'})
    )
    grp = grp.merge(example_kpis, on='cmdb_id', how='left')
    grp['node_tag'] = node_tag
    grp['earliest_anomaly_timestamp'] = grp['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    grp = grp[['node_tag','cmdb_id','number_of_distinct_flagged_kpis','total_anomalous_points_in_window','earliest_anomaly_timestamp','example_kpis']]
    node_rows.append(grp)

if node_rows:
    node_table = pd.concat(node_rows, axis=0, ignore_index=True).sort_values('total_anomalous_points_in_window', ascending=False).head(20).reset_index(drop=True)
else:
    node_table = pd.DataFrame(columns=['node_tag','cmdb_id','number_of_distinct_flagged_kpis','total_anomalous_points_in_window','earliest_anomaly_timestamp','example_kpis'])

# --- 2) Service correlation table for cartservice, checkoutservice, adservice ---
services = ['cartservice','checkoutservice','adservice']
service_rows = []

# Use metric_breach_full if available to get most severe metric breach per service; otherwise use metric_anom_agg as fallback
metric_breach_src = metric_breach_full if 'metric_breach_full' in globals() else (metric_anom_agg.merge(metric_thresh, on=['cmdb_id','kpi_name'], how='left') if 'metric_thresh' in globals() else None)

for svc in services:
    # cmdb ids involved (from metric and trace anomalies and from raw metric_win/trace_win)
    cmdb_metric = pd.Index(metric_anom_agg['cmdb_id'].astype(str).unique()) if 'metric_anom_agg' in globals() else pd.Index([])
    cmdb_trace = pd.Index(trace_anom_agg['cmdb_id'].astype(str).unique()) if 'trace_anom_agg' in globals() else pd.Index([])
    # match prefix e.g., cartservice or cartservice-0 etc.
    cmdbs = sorted([c for c in pd.Index(cmdb_metric.union(cmdb_trace)) if str(c).startswith(svc)])
    cmdbs_involved = cmdbs[:6]  # up to 6

    # earliest trace anomaly timestamp for this service
    if 'trace_anom_agg' in globals():
        mask_trace = trace_anom_agg['cmdb_id'].astype(str).str.startswith(svc)
        if mask_trace.any():
            earliest_trace_ts = pd.to_datetime(trace_anom_agg.loc[mask_trace,'earliest_anomaly_timestamp'].min())
            # ensure tz formatting
            try:
                earliest_trace_ts = pd.to_datetime(trace_anom_agg.loc[mask_trace,'earliest_anomaly_timestamp']).min()
                # parse and reformat to tz
                earliest_trace_ts_dt = pd.to_datetime(earliest_trace_ts, format='%Y-%m-%d %H:%M:%S %Z', errors='coerce')
                if pd.isna(earliest_trace_ts_dt):
                    # try naive parse then localize
                    earliest_trace_ts_dt = pd.to_datetime(str(earliest_trace_ts)).tz_localize(tz)
                else:
                    earliest_trace_ts_dt = earliest_trace_ts_dt.tz_convert(tz) if earliest_trace_ts_dt.tzinfo else earliest_trace_ts_dt.tz_localize(tz)
                earliest_trace_ts_str = earliest_trace_ts_dt.strftime('%Y-%m-%d %H:%M:%S %Z')
            except Exception:
                earliest_trace_ts_str = None
        else:
            earliest_trace_ts_str = None
    else:
        earliest_trace_ts_str = None

    # earliest metric fault timestamp for this service
    if 'metric_anom_agg' in globals():
        mask_metric = metric_anom_agg['cmdb_id'].astype(str).str.startswith(svc)
        if mask_metric.any():
            try:
                val = metric_anom_agg.loc[mask_metric,'earliest_anomaly_timestamp'].min()
                # parse & format
                dt = pd.to_datetime(val, format='%Y-%m-%d %H:%M:%S %Z', errors='coerce')
                if pd.isna(dt):
                    dt = pd.to_datetime(str(val)).tz_localize(tz)
                else:
                    dt = dt.tz_convert(tz) if dt.tzinfo else dt.tz_localize(tz)
                earliest_metric_ts_str = dt.strftime('%Y-%m-%d %H:%M:%S %Z')
            except Exception:
                earliest_metric_ts_str = None
        else:
            earliest_metric_ts_str = None
    else:
        earliest_metric_ts_str = None

    # most severe metric for this service (highest breach_factor)
    most_severe = None
    most_severe_ts = None
    most_severe_breach = None
    if metric_breach_src is not None:
        # ensure metric_breach_src has 'cmdb_id','kpi_name','breach_factor'
        src = metric_breach_src.copy()
        if 'breach_factor' not in src.columns and 'global_p95' in src.columns and 'in_window_max_value' in src.columns:
            # compute breach_factor if absent
            def bf_calc(r):
                if pd.notna(r.get('in_window_max_value')) and pd.notna(r.get('global_p95')) and r['global_p95']>0 and r['in_window_max_value']>r['global_p95']:
                    return r['in_window_max_value']/r['global_p95']
                if pd.notna(r.get('in_window_min_value')) and pd.notna(r.get('global_p5')) and r['in_window_min_value']>0 and r['in_window_min_value']<r['global_p5']:
                    return r['global_p5']/r['in_window_min_value']
                return 0.0
            src['breach_factor'] = src.apply(bf_calc, axis=1)
        mask_src = src['cmdb_id'].astype(str).str.startswith(svc)
        src_svc = src[mask_src].copy()
        if not src_svc.empty:
            src_svc = src_svc.sort_values('breach_factor', ascending=False)
            top = src_svc.iloc[0]
            most_severe = f"{top['cmdb_id']}||{top.get('kpi_name', top.get('trace_name', ''))}"
            most_severe_breach = float(top['breach_factor'])
            # find earliest anomaly timestamp from metric_anom_agg for that series if available
            if 'metric_anom_agg' in globals():
                match = metric_anom_agg[(metric_anom_agg['cmdb_id']==top['cmdb_id']) & (metric_anom_agg['kpi_name']==top.get('kpi_name'))]
                if not match.empty:
                    # take min earliest_anomaly_timestamp
                    ts_val = match['earliest_anomaly_timestamp'].min()
                    most_severe_ts = ts_val
                else:
                    most_severe_ts = None
    # assemble row
    service_rows.append({
        'service_name': svc,
        'cmdb_ids_involved': cmdbs_involved,
        'earliest_trace_anomaly_ts': earliest_trace_ts_str,
        'earliest_metric_fault_ts': earliest_metric_ts_str,
        'most_severe_metric': most_severe,
        'most_severe_metric_earliest_ts': most_severe_ts,
        'most_severe_breach_ratio': most_severe_breach
    })

service_table = pd.DataFrame(service_rows)

# Keep outputs compact per instructions
node_table, service_table

```
Out[7]:
```
```python
# Summarize localization evidence in plain English using existing kernel tables (node_table, service_table).
summary = (
    "Localization evidence (compact):\n\n"
    "1) Node-to-service anomalies:\n"
    "- node-5 shows broad, systemic anomalies across many services. Examples from the table:\n"
    "  * emailservice2-0: 64 distinct node-5 KPIs flagged, 1703 anomalous points, earliest 2022-03-21 00:00:00 CST.\n"
    "  * emailservice-0 / emailservice-2 / emailservice-1 / recommendationservice-* / adservice-* / paymentservice-* / shippingservice-* / frontend-0\n"
    "    all show dozens of distinct node-5 KPIs flagged and >1500 anomalous points, earliest 2022-03-21 00:00:00 CST.\n"
    "- node-3 did not appear in this node-tagged summary (node-5 is the dominant hotspot in this output).\n\n"
    "2) Service correlation (cartservice, checkoutservice, adservice):\n"
    "- cartservice: involved cmdb_ids include cartservice, cartservice-0/1/2; earliest trace anomaly 2022-03-21 00:00:00 CST; earliest metric fault 00:00:00 CST.\n"
    "  Most severe metric: cartservice-2 || container.node-5.container_network_receive... (breach_ratio ≈ 12.59), earliest ~2022-03-21 00:04:00 CST.\n"
    "- checkoutservice: involved cmdb_ids include checkoutservice and pods; earliest trace and metric anomalies at 2022-03-21 00:00:00 CST.\n"
    "  Most severe metric: checkoutservice-2 || container.node-5.container_cpu_usage_seconds (breach_ratio ≈ 24.18), earliest 2022-03-21 00:00:00 CST.\n"
    "- adservice: involved cmdb_ids include adservice and pods; earliest trace and metric anomalies at 2022-03-21 00:00:00 CST.\n"
    "  Most severe metric: adservice-1 || container.node-5.container_network_receive... (breach_ratio ≈ 13.76), earliest 2022-03-21 00:00:00 CST.\n\n"
    "Overall takeaway and next actions:\n"
    "- Strong evidence points to node-5 as a centralized hotspot (many containers on node-5 show anomalous KPI activity beginning at 00:00). Investigate node-5 (network, CPU throttle, and container networking receive) first.\n"
    "- Checkoutservice-2 shows the single largest breach_ratio; investigate its CPU/network on node-5 and correlate with cart↔checkout traces (early-window failures).\n"
    "- Adservice and frontend traces show elevated latencies in a later cluster — correlate with the node-5 network spikes and frontend error logs.\n"
    "- Recommended immediate steps: (a) check node-5 host metrics (network, CPU throttling), (b) inspect checkoutservice-2 and cartservice traces around 00:00–00:05, (c) examine frontend error logs for corroborating errors.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   node_tag                  cmdb_id  number_of_distinct_flagged_kpis  total_anomalous_points_in_window earliest_anomaly_timestamp                                       example_kpis
0    node-5          emailservice2-0                               64                              1703    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
1    node-5           emailservice-0                               64                              1700    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
2    node-5           emailservice-2                               64                              1685    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
3    node-5           emailservice-1                               64                              1677    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
4    node-5  recommendationservice-1                               64                              1673    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
5    node-5  recommendationservice-0                               64                              1667    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_periods, c...
6    node-5  recommendationservice-2                               64                              1663    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
7    node-5              adservice-1                               64                              1649    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
8    node-5              adservice-2                               64                              1646    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
9    node-5              adservice-0                               64                              1618    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_periods, c...
10   node-5         paymentservice-2                               64                              1594    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_periods, c...
11   node-5         paymentservice-0                               64                              1592    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_periods, c...
12   node-5       shippingservice2-0                               64                              1564    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
13   node-5  productcatalogservice-1                               64                              1562    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_periods, c...
14   node-5        shippingservice-2                               64                              1560    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
15   node-5        shippingservice-0                               64                              1559    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_periods, c...
16   node-5  productcatalogservice-2                               64                              1557    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
17   node-5  productcatalogservice-0                               64                              1556    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
18   node-5        shippingservice-1                               64                              1554    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...
19   node-5               frontend-0                               64                              1537    2022-03-21 00:00:00 CST  [container.node-5.container_cpu_cfs_throttled_...,       service_name                                  cmdb_ids_involved earliest_trace_anomaly_ts earliest_metric_fault_ts                                 most_severe_metric most_severe_metric_earliest_ts  most_severe_breach_ratio
0      cartservice  [cartservice, cartservice-0, cartservice-1, ca...   2022-03-21 00:00:00 CST  2022-03-21 00:00:00 CST  cartservice-2||container.node-5.container_netw...        2022-03-21 00:04:00 CST                 12.591209
1  checkoutservice  [checkoutservice, checkoutservice-0, checkouts...   2022-03-21 00:00:00 CST  2022-03-21 00:00:00 CST  checkoutservice-2||container.node-5.container_...        2022-03-21 00:00:00 CST                 24.182514
2        adservice  [adservice, adservice-0, adservice-1, adservic...   2022-03-21 00:00:00 CST  2022-03-21 00:00:00 CST  adservice-1||container.node-5.container_networ...        2022-03-21 00:00:00 CST                 13.755274)```
```

In [ ]:
# In[8]:

import pandas as pd
import re

# Reuse existing metric_df if present, else load
metric_df = metric_df if 'metric_df' in globals() else pd.read_csv("metric.csv")

# Ensure kpi_name is string
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)

# Filter KPI rows that contain the exact substring 'container.node-'
mask_node_kpis = metric_df['kpi_name'].str.contains('container.node-', na=False)
node_kpi_rows = metric_df[mask_node_kpis].copy()

# Function to extract unique node tags like 'node-5' from a kpi_name
def extract_node_tags(kpi_name):
    return re.findall(r'node-\d+', kpi_name)

# Aggregate per cmdb_id
grouped = (
    node_kpi_rows.groupby('cmdb_id')['kpi_name']
    .agg(lambda s: list(pd.Index(s.dropna().unique())))
    .reset_index()
    .rename(columns={'kpi_name':'node_kpis'})
)

# Compute nodes_observed, counts, and sample_kpis
def nodes_observed_from_list(kpi_list):
    tags = set()
    for k in kpi_list:
        tags.update(extract_node_tags(k))
    return sorted(tags)

grouped['nodes_observed'] = grouped['node_kpis'].apply(nodes_observed_from_list)
grouped['n_distinct_node_kpis'] = grouped['node_kpis'].apply(len)
grouped['sample_node_kpis'] = grouped['node_kpis'].apply(lambda lst: lst[:5])

# Select and order columns compactly
mapping_table_all = grouped[['cmdb_id','nodes_observed','n_distinct_node_kpis','sample_node_kpis']].copy()
# Sort by number of distinct node kpis desc, then cmdb_id
mapping_table_all = mapping_table_all.sort_values(['n_distinct_node_kpis','cmdb_id'], ascending=[False,True]).reset_index(drop=True)

# Top 50 rows (bounded)
mapping_table = mapping_table_all.head(50).copy()

# Focused subsection for services prefixes (include any pod suffix variants)
service_prefixes = ['cartservice','checkoutservice','adservice','frontend']
# find cmdb_ids that start with any of these prefixes
def matches_prefix(cmdb):
    return any(str(cmdb).startswith(pref) for pref in service_prefixes)

focused_table = mapping_table_all[mapping_table_all['cmdb_id'].apply(matches_prefix)].copy()

# Ensure focused_table includes entries even if not in top 50 (deduplicated)
# If some focused cmdb_ids are not in top 50, append them from the full mapping table
focused_ids = set(focused_table['cmdb_id'].tolist())
missing_focused = mapping_table_all[~mapping_table_all['cmdb_id'].isin(focused_ids) & mapping_table_all['cmdb_id'].apply(matches_prefix)]
if not missing_focused.empty:
    focused_table = pd.concat([focused_table, missing_focused]).reset_index(drop=True)

# Keep outputs compact: mapping_table (<=50 rows) and focused_table (only requested prefixes)
mapping_table, focused_table

```
Out[8]:
```
```python
# Plain-English summary of findings from scanning metric.csv for 'container.node-' KPIs.
summary = (
    "Summary of 'container.node-' KPI mapping (compact):\n\n"
    "1) Scope and overall pattern:\n"
    "- Found 42 cmdb_id entries with KPIs containing the exact substring 'container.node-'.\n"
    "- The vast majority of affected containers report metrics referencing node-5 (nodes_observed = ['node-5']). A few pods reference node-6.\n"
    "- Many services have a large set of node-related KPIs (n_distinct_node_kpis ≈ 64 for most service pods), e.g., container.node-5.container_cpu_cfs_periods, container_cpu_cfs_throttled, container_network_receive_..., etc.\n\n"
    "2) Notable counts and examples:\n"
    "- Typical entries (adservice-0/1/2, cartservice-0/1/2, checkoutservice-0/1/2, frontend-0/1/2, paymentservice-*, shippingservice-*, productcatalogservice-*) show nodes_observed = ['node-5'] and ~64 distinct node-related KPIs.\n"
    "- A small subset (e.g., adservice2-0, cartservice2-0, frontend2-0) reference node-6.\n"
    "- Some specialized components (redis-cart-0, redis-cart2-0) list far fewer node KPIs (n_distinct_node_kpis = 8) and have example network-related node KPIs in their sample list.\n\n"
    "3) Focused services (cartservice, checkoutservice, adservice, frontend):\n"
    "- All these services (including their pod variants -0/-1/-2) appear in the focused subsection and mostly map to node-5 KPIs.\n"
    "- This strengthens the observation that node-5 is the common host for many affected service pods in this dataset.\n\n"
    "Actionable takeaways:\n"
    "- Node-5 is the primary hotspot: investigate host-level resource and network issues on node-5 (CPU throttling, network_receive spikes, disk/io metrics) because many containers across services expose node-5 KPIs and show anomalies.\n"
    "- Also inspect the smaller set of pods on node-6 (some replica2 pods) as a secondary check.\n"
    "- Use the provided mapping table (stored in kernel variable) to pick representative KPI names (sample_node_kpis) per cmdb_id when drilling into logs/metrics on node-5.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id nodes_observed  n_distinct_node_kpis                                   sample_node_kpis
0          adservice-0       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
1          adservice-1       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
2          adservice-2       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
3         adservice2-0       [node-6]                    64  [container.node-6.container_cpu_cfs_periods, c...
4        cartservice-0       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
..                 ...            ...                   ...                                                ...
37   shippingservice-2       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
38  shippingservice2-0       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
39    paymentservice-1       [node-5]                    60  [container.node-5.container_cpu_cfs_periods, c...
40        redis-cart-0       [node-5]                     8  [container.node-5.container_network_receive_MB...
41       redis-cart2-0       [node-5]                     8  [container.node-5.container_network_receive_MB...

[42 rows x 4 columns],                cmdb_id nodes_observed  n_distinct_node_kpis                                   sample_node_kpis
0          adservice-0       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
1          adservice-1       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
2          adservice-2       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
3         adservice2-0       [node-6]                    64  [container.node-6.container_cpu_cfs_periods, c...
4        cartservice-0       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
5        cartservice-1       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
6        cartservice-2       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
7       cartservice2-0       [node-6]                    64  [container.node-6.container_cpu_cfs_periods, c...
8    checkoutservice-0       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
9    checkoutservice-1       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
10   checkoutservice-2       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
11  checkoutservice2-0       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
20          frontend-0       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
21          frontend-1       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
22          frontend-2       [node-5]                    64  [container.node-5.container_cpu_cfs_periods, c...
23         frontend2-0       [node-6]                    64  [container.node-6.container_cpu_cfs_periods, c...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```